# Person 3 stage 3: GILE input-layout fix (C) and person-style augmentation (B) — UCI HAR, T4
Plan and success rule fixed in PREDECLARED_PLAN.md before any result. **Run cells one at a time.** After each seed cell,
wait until its zip has downloaded before starting the next cell. No Google Drive, no account data.

In [ ]:
# 1. GPU check
import subprocess, torch, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))

In [ ]:
# 2. Write embedded files, verify SHA-256
import base64, hashlib, json, os
PKG = '/content/p3'
PAYLOAD = json.loads('{"code/data_preprocess_ucihar.py": {"sha256": "5c81526c9c126370de0574bb60c04907c569719b8cabc5a592ce45d461863f70", "b64": "IyBlbmNvZGluZz11dGYtOA0KaW1wb3J0IG9zDQppbXBvcnQgbnVtcHkgYXMgbnANCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YXNldCwgRGF0YUxvYWRlcg0KZnJvbSB0b3JjaHZpc2lvbiBpbXBvcnQgdHJhbnNmb3Jtcw0KaW1wb3J0IHRvcmNoDQpmcm9tIHNsaWRpbmdfd2luZG93IGltcG9ydCBzbGlkaW5nX3dpbmRvdw0KZnJvbSB1dGlscyBpbXBvcnQgZ2V0X3NhbXBsZV93ZWlnaHRzDQppbXBvcnQgc2NpcHkuaW8NCmltcG9ydCBwaWNrbGUgYXMgY3ANCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IFN0cmF0aWZpZWRTaHVmZmxlU3BsaXQNCg0KDQojIFRoaXMgaXMgZm9yIHBhcnNpbmcgdGhlIFggZGF0YSwgeW91IGNhbiBpZ25vcmUgaXQgaWYgeW91IGRvIG5vdCBuZWVkIHByZXByb2Nlc3NpbmcNCmRlZiBmb3JtYXRfZGF0YV94KGRhdGFmaWxlKToNCiAgICB4X2RhdGEgPSBOb25lDQogICAgZm9yIGl0ZW0gaW4gZGF0YWZpbGU6DQogICAgICAgIGl0ZW1fZGF0YSA9IG5wLmxvYWR0eHQoaXRlbSwgZHR5cGU9ZmxvYXQpICAjIFBlcnNvbiAzOiBucC5mbG9hdCByZW1vdmVkIGluIG51bXB5Pj0xLjI0IChzYW1lIGR0eXBlKQ0KICAgICAgICBpZiB4X2RhdGEgaXMgTm9uZToNCiAgICAgICAgICAgIHhfZGF0YSA9IG5wLnplcm9zKChsZW4oaXRlbV9kYXRhKSwgMSkpDQogICAgICAgIHhfZGF0YSA9IG5wLmhzdGFjaygoeF9kYXRhLCBpdGVtX2RhdGEpKQ0KICAgIHhfZGF0YSA9IHhfZGF0YVs6LCAxOl0NCiAgICBwcmludCgneF9kYXRhLnNoYXBlOicsIHhfZGF0YS5zaGFwZSkNCiAgICBYID0gTm9uZQ0KICAgIGZvciBpIGluIHJhbmdlKGxlbih4X2RhdGEpKToNCiAgICAgICAgcm93ID0gbnAuYXNhcnJheSh4X2RhdGFbaSwgOl0pDQogICAgICAgIHJvdyA9IHJvdy5yZXNoYXBlKDksIDEyOCkuVA0KICAgICAgICBpZiBYIGlzIE5vbmU6DQogICAgICAgICAgICBYID0gbnAuemVyb3MoKGxlbih4X2RhdGEpLCAxMjgsIDkpKQ0KICAgICAgICBYW2ldID0gcm93DQogICAgcHJpbnQoJ1guc2hhcGU6JywgWC5zaGFwZSkNCiAgICByZXR1cm4gWA0KDQpkZWYgZm9ybWF0X2RhdGFfeShkYXRhZmlsZSk6DQogICAgZGF0YSA9IG5wLmxvYWR0eHQoZGF0YWZpbGUsIGR0eXBlPWludCkgLSAxICAjIFBlcnNvbiAzOiBucC5pbnQgcmVtb3ZlZCBpbiBudW1weT49MS4yNCAoc2FtZSBkdHlwZSkNCiAgICByZXR1cm4gZGF0YQ0KDQoNCmRlZiBsb2FkX2RvbWFpbl9kYXRhKGRvbWFpbl9pZHgpOg0KICAgICIiIiB0byBsb2FkIGFsbCB0aGUgZGF0YSBmcm9tIHRoZSBzcGVjaWZpYyBkb21haW4NCiAgICA6cGFyYW0gZG9tYWluX2lkeDoNCiAgICA6cmV0dXJuOiBYIGFuZCB5IGRhdGEgb2YgdGhlIGVudGlyZSBkb21haW4NCiAgICAiIiINCiAgICBkYXRhX2RpciA9ICcuL2RhdGEvdWNpaGFyLycNCiAgICBzYXZlZF9maWxlbmFtZSA9ICd1Y2loYXJfZG9tYWluXycgKyBkb21haW5faWR4ICsgJ193ZC5kYXRhJyAjIHdpdGggZG9tYWluIGxhYmVsDQoNCiAgICBpZiBvcy5wYXRoLmlzZmlsZShkYXRhX2RpciArIHNhdmVkX2ZpbGVuYW1lKSA9PSBUcnVlOg0KICAgICAgICBkYXRhID0gbnAubG9hZChkYXRhX2RpciArIHNhdmVkX2ZpbGVuYW1lLCBhbGxvd19waWNrbGU9VHJ1ZSkNCiAgICAgICAgWCA9IGRhdGFbMF1bMF0NCiAgICAgICAgeSA9IGRhdGFbMF1bMV0NCiAgICAgICAgZCA9IGRhdGFbMF1bMl0NCiAgICBlbHNlOg0KICAgICAgICBzdHJfZm9sZGVyID0gJy4vZGF0YS91Y2loYXIvJw0KICAgICAgICBJTlBVVF9TSUdOQUxfVFlQRVMgPSBbDQogICAgICAgICAgICAiYm9keV9hY2NfeF8iLA0KICAgICAgICAgICAgImJvZHlfYWNjX3lfIiwNCiAgICAgICAgICAgICJib2R5X2FjY196XyIsDQogICAgICAgICAgICAiYm9keV9neXJvX3hfIiwNCiAgICAgICAgICAgICJib2R5X2d5cm9feV8iLA0KICAgICAgICAgICAgImJvZHlfZ3lyb196XyIsDQogICAgICAgICAgICAidG90YWxfYWNjX3hfIiwNCiAgICAgICAgICAgICJ0b3RhbF9hY2NfeV8iLA0KICAgICAgICAgICAgInRvdGFsX2FjY196XyINCiAgICAgICAgXQ0KICAgICAgICBzdHJfdHJhaW5fZmlsZXMgPSBbc3RyX2ZvbGRlciArICd0cmFpbi8nICsgJ0luZXJ0aWFsU2lnbmFscy8nICsgaXRlbSArICd0cmFpbi50eHQnIGZvciBpdGVtIGluDQogICAgICAgICAgICAgICAgICAgICAgICAgICBJTlBVVF9TSUdOQUxfVFlQRVNdDQogICAgICAgIHN0cl90ZXN0X2ZpbGVzID0gW3N0cl9mb2xkZXIgKyAndGVzdC8nICsgJ0luZXJ0aWFsU2lnbmFscy8nICsgaXRlbSArICd0ZXN0LnR4dCcgZm9yIGl0ZW0gaW4gSU5QVVRfU0lHTkFMX1RZUEVTXQ0KICAgICAgICBzdHJfdHJhaW5feSA9IHN0cl9mb2xkZXIgKyAndHJhaW4veV90cmFpbi50eHQnDQogICAgICAgIHN0cl90ZXN0X3kgPSBzdHJfZm9sZGVyICsgJ3Rlc3QveV90ZXN0LnR4dCcNCg0KICAgICAgICBzdHJfdHJhaW5faWQgPSBzdHJfZm9sZGVyICsgJ3RyYWluL3N1YmplY3RfdHJhaW4udHh0Jw0KICAgICAgICBzdHJfdGVzdF9pZCA9IHN0cl9mb2xkZXIgKyAndGVzdC9zdWJqZWN0X3Rlc3QudHh0Jw0KDQogICAgICAgIFhfdHJhaW4gPSBmb3JtYXRfZGF0YV94KHN0cl90cmFpbl9maWxlcykNCiAgICAgICAgWF90ZXN0ID0gZm9ybWF0X2RhdGFfeChzdHJfdGVzdF9maWxlcykNCiAgICAgICAgWV90cmFpbiA9IGZvcm1hdF9kYXRhX3koc3RyX3RyYWluX3kpDQogICAgICAgIFlfdGVzdCA9IGZvcm1hdF9kYXRhX3koc3RyX3Rlc3RfeSkNCiAgICAgICAgaWRfdHJhaW4gPSBmb3JtYXRfZGF0YV95KA0KICAgICAgICAgICAgc3RyX3RyYWluX2lkKSAgIyBvcmlnaW46IGFycmF5KFsgMCwgIDIsICA0LCAgNSwgIDYsICA3LCAxMCwgMTMsIDE0LCAxNSwgMTYsIDE4LCAyMCwgMjEsIDIyLCAyNCwgMjUsIDI2LCAyNywgMjgsIDI5XSkNCiAgICAgICAgaWRfdGVzdCA9IGZvcm1hdF9kYXRhX3koc3RyX3Rlc3RfaWQpICAjIG9yaWdpbjogYXJyYXkoWyAxLCAgMywgIDgsICA5LCAxMSwgMTIsIDE3LCAxOSwgMjNdKQ0KDQogICAgICAgIFhfYWxsID0gbnAuY29uY2F0ZW5hdGUoKFhfdHJhaW4sIFhfdGVzdCksIGF4aXM9MCkNCiAgICAgICAgeV9hbGwgPSBucC5jb25jYXRlbmF0ZSgoWV90cmFpbiwgWV90ZXN0KSwgYXhpcz0wKQ0KICAgICAgICBpZF9hbGwgPSBucC5jb25jYXRlbmF0ZSgoaWRfdHJhaW4sIGlkX3Rlc3QpLCBheGlzPTApDQoNCiAgICAgICAgcHJpbnQoJ1xuUHJvY2Vzc2luZyBkb21haW4gezB9IGZpbGVzLi4uXG4nLmZvcm1hdChkb21haW5faWR4KSkNCiAgICAgICAgdGFyZ2V0X2lkeCA9IG5wLndoZXJlKGlkX2FsbCA9PSBpbnQoZG9tYWluX2lkeCkpDQogICAgICAgIFggPSBYX2FsbFt0YXJnZXRfaWR4XQ0KICAgICAgICB5ID0geV9hbGxbdGFyZ2V0X2lkeF0NCiAgICAgICAgZCA9IG5wLmZ1bGwoeS5zaGFwZSwgaW50KGRvbWFpbl9pZHgpLCBkdHlwZT1pbnQpDQogICAgICAgIHByaW50KCdcblByb2Nlc3NpbmcgZG9tYWluIHswfSBmaWxlcyB8IFg6IHsxfSB5OiB7Mn0gZDp7M30gXG4nLmZvcm1hdChkb21haW5faWR4LCBYLnNoYXBlLCB5LnNoYXBlLCBkLnNoYXBlKSkNCg0KICAgICAgICBvYmogPSBbKFgsIHksIGQpXQ0KICAgICAgICAjIGZpbGUgZnVuY3Rpb24gaXMgbm90IHN1cHBvcnRlZCBpbiBweXRob24zLCB1c2Ugb3BlbiBpbnN0ZWFkDQogICAgICAgIGYgPSBvcGVuKG9zLnBhdGguam9pbihkYXRhX2Rpciwgc2F2ZWRfZmlsZW5hbWUpLCAnd2InKQ0KICAgICAgICBjcC5kdW1wKG9iaiwgZiwgcHJvdG9jb2w9Y3AuSElHSEVTVF9QUk9UT0NPTCkNCiAgICAgICAgZi5jbG9zZSgpDQoNCiAgICByZXR1cm4gWCwgeSwgZA0KDQpjbGFzcyBkYXRhX2xvYWRlcl91Y2loYXIoRGF0YXNldCk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNhbXBsZXMsIGxhYmVscywgZG9tYWlucywgdCk6DQogICAgICAgIHNlbGYuc2FtcGxlcyA9IHNhbXBsZXMNCiAgICAgICAgc2VsZi5sYWJlbHMgPSBsYWJlbHMNCiAgICAgICAgc2VsZi5kb21haW5zID0gZG9tYWlucw0KICAgICAgICBzZWxmLlQgPSB0DQoNCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaW5kZXgpOg0KICAgICAgICBzYW1wbGUsIHRhcmdldCwgZG9tYWluID0gc2VsZi5zYW1wbGVzW2luZGV4XSwgc2VsZi5sYWJlbHNbaW5kZXhdLCBzZWxmLmRvbWFpbnNbaW5kZXhdDQogICAgICAgIHNhbXBsZSA9IHNlbGYuVChzYW1wbGUpDQogICAgICAgIHJldHVybiBucC50cmFuc3Bvc2Uoc2FtcGxlLCAoMSwgMCwgMikpLCB0YXJnZXQsIGRvbWFpbg0KDQogICAgZGVmIF9fbGVuX18oc2VsZik6DQogICAgICAgIHJldHVybiBsZW4oc2VsZi5zYW1wbGVzKQ0KDQoNCmRlZiBwcmVwX2RvbWFpbnNfdWNpaGFyKGFyZ3MsIFNMSURJTkdfV0lORE9XX0xFTj0wLCBTTElESU5HX1dJTkRPV19TVEVQPTApOg0KDQogICAgc291cmNlX2RvbWFpbl9saXN0ID0gWycwJywgJzEnLCAnMicsICczJywgJzQnXQ0KICAgIHNvdXJjZV9kb21haW5fbGlzdC5yZW1vdmUoYXJncy50YXJnZXRfZG9tYWluKQ0KDQogICAgIyBzb3VyY2UgZG9tYWluIGRhdGEgcHJlcA0KICAgIHNvdXJjZV9sb2FkZXJzID0gW10NCiAgICBmb3Igc291cmNlX2RvbWFpbiBpbiBzb3VyY2VfZG9tYWluX2xpc3Q6DQogICAgICAgIHByaW50KCdzb3VyY2VfZG9tYWluOicsIHNvdXJjZV9kb21haW4pDQogICAgICAgIHgsIHksIGQgPSBsb2FkX2RvbWFpbl9kYXRhKHNvdXJjZV9kb21haW4pDQogICAgICAgIHggPSBucC50cmFuc3Bvc2UoeC5yZXNoYXBlKCgtMSwgMSwgMTI4LCA5KSksICgwLCAyLCAxLCAzKSkNCiAgICAgICAgdW5pcXVlX3ksIGNvdW50c195ID0gbnAudW5pcXVlKHksIHJldHVybl9jb3VudHM9VHJ1ZSkNCiAgICAgICAgd2VpZ2h0cyA9IDEwMC4wIC8gdG9yY2guVGVuc29yKGNvdW50c195KQ0KICAgICAgICB3ZWlnaHRzID0gd2VpZ2h0cy5kb3VibGUoKQ0KICAgICAgICBzYW1wbGVfd2VpZ2h0cyA9IGdldF9zYW1wbGVfd2VpZ2h0cyh5LCB3ZWlnaHRzKQ0KICAgICAgICBzYW1wbGVyID0gdG9yY2gudXRpbHMuZGF0YS5zYW1wbGVyLldlaWdodGVkUmFuZG9tU2FtcGxlcih3ZWlnaHRzPXNhbXBsZV93ZWlnaHRzLCBudW1fc2FtcGxlcz1sZW4oc2FtcGxlX3dlaWdodHMpLCByZXBsYWNlbWVudD1UcnVlKQ0KICAgICAgICB0cmFuc2Zvcm0gPSB0cmFuc2Zvcm1zLkNvbXBvc2UoWw0KICAgICAgICAgICAgdHJhbnNmb3Jtcy5Ub1RlbnNvcigpLA0KICAgICAgICAgICAgdHJhbnNmb3Jtcy5Ob3JtYWxpemUobWVhbj0oMCwgMCwgMCwgMCwgMCwgMCwgMCwgMCwgMCksIHN0ZD0oMSwgMSwgMSwgMSwgMSwgMSwgMSwgMSwgMSkpDQogICAgICAgIF0pDQogICAgICAgIGRhdGFfc2V0ID0gZGF0YV9sb2FkZXJfdWNpaGFyKHgsIHksIGQsIHRyYW5zZm9ybSkNCiAgICAgICAgc291cmNlX2xvYWRlciA9IERhdGFMb2FkZXIoZGF0YV9zZXQsIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlLCBkcm9wX2xhc3Q9VHJ1ZSwgc2FtcGxlcj1zYW1wbGVyKQ0KICAgICAgICBwcmludCgnc291cmNlX2xvYWRlciBiYXRjaDogJywgbGVuKHNvdXJjZV9sb2FkZXIpKQ0KICAgICAgICBzb3VyY2VfbG9hZGVycy5hcHBlbmQoc291cmNlX2xvYWRlcikNCg0KICAgICMgdGFyZ2V0IGRvbWFpbiBkYXRhIHByZXANCiAgICBwcmludCgndGFyZ2V0X2RvbWFpbjonLCBhcmdzLnRhcmdldF9kb21haW4pDQogICAgeCwgeSwgZCA9IGxvYWRfZG9tYWluX2RhdGEoYXJncy50YXJnZXRfZG9tYWluKQ0KICAgIHggPSBucC50cmFuc3Bvc2UoeC5yZXNoYXBlKCgtMSwgMSwgMTI4LCA5KSksICgwLCAyLCAxLCAzKSkNCiAgICBkYXRhX3NldCA9IGRhdGFfbG9hZGVyX3VjaWhhcih4LCB5LCBkLCB0cmFuc2Zvcm0pDQogICAgdGFyZ2V0X2xvYWRlciA9IERhdGFMb2FkZXIoZGF0YV9zZXQsIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlKQ0KICAgIHByaW50KCd0YXJnZXRfbG9hZGVyIGJhdGNoOiAnLCBsZW4odGFyZ2V0X2xvYWRlcikpDQogICAgcmV0dXJuIHNvdXJjZV9sb2FkZXJzLCB0YXJnZXRfbG9hZGVyDQoNCg=="}, "code/model_GILE_ucihar.py": {"sha256": "e0729904ceb905f13903ceecaff59e8a8b083f086d0cade0261eccea0c4859bc", "b64": "..."}, "code/sliding_window.py": {"sha256": "7c4dfd680eba398f61d81dc4666f6c2484e3211a37769aae6a100a7c2df360a2", "b64": "IyBBZGFwdGVkIEZyb206IGh0dHBzOi8vZ2l0aHViLmNvbS9zdXNzZXh3ZWFybGFiL0RlZXBDb252TFNUTQ0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmZyb20gbnVtcHkubGliLnN0cmlkZV90cmlja3MgaW1wb3J0IGFzX3N0cmlkZWQgYXMgYXN0DQoNCmRlZiBub3JtX3NoYXBlKHNoYXBlKToNCiAgICAnJycNCiAgICBOb3JtYWxpemUgbnVtcHkgYXJyYXkgc2hhcGVzIHNvIHRoZXkncmUgYWx3YXlzIGV4cHJlc3NlZCBhcyBhIHR1cGxlLA0KICAgIGV2ZW4gZm9yIG9uZS1kaW1lbnNpb25hbCBzaGFwZXMuDQoNCiAgICBQYXJhbWV0ZXJzDQogICAgICAgIHNoYXBlIC0gYW4gaW50LCBvciBhIHR1cGxlIG9mIGludHMNCg0KICAgIFJldHVybnMNCiAgICAgICAgYSBzaGFwZSB0dXBsZQ0KICAgICcnJw0KICAgIHRyeToNCiAgICAgICAgaSA9IGludChzaGFwZSkNCiAgICAgICAgcmV0dXJuIChpLCkNCiAgICBleGNlcHQgVHlwZUVycm9yOg0KICAgICAgICAjIHNoYXBlIHdhcyBub3QgYSBudW1iZXINCiAgICAgICAgcGFzcw0KDQogICAgdHJ5Og0KICAgICAgICB0ID0gdHVwbGUoc2hhcGUpDQogICAgICAgIHJldHVybiB0DQogICAgZXhjZXB0IFR5cGVFcnJvcjoNCiAgICAgICAgIyBzaGFwZSB3YXMgbm90IGl0ZXJhYmxlDQogICAgICAgIHBhc3MNCg0KICAgIHJhaXNlIFR5cGVFcnJvcignc2hhcGUgbXVzdCBiZSBhbiBpbnQsIG9yIGEgdHVwbGUgb2YgaW50cycpDQoNCmRlZiBzbGlkaW5nX3dpbmRvdyhhLHdzLHNzID0gTm9uZSxmbGF0dGVuID0gVHJ1ZSk6DQogICAgJycnDQogICAgUmV0dXJuIGEgc2xpZGluZyB3aW5kb3cgb3ZlciBhIGluIGFueSBudW1iZXIgb2YgZGltZW5zaW9ucw0KDQogICAgUGFyYW1ldGVyczoNCiAgICAgICAgYSAgLSBhbiBuLWRpbWVuc2lvbmFsIG51bXB5IGFycmF5DQogICAgICAgIHdzIC0gYW4gaW50IChhIGlzIDFEKSBvciB0dXBsZSAoYSBpcyAyRCBvciBncmVhdGVyKSByZXByZXNlbnRpbmcgdGhlIHNpemUNCiAgICAgICAgICAgICBvZiBlYWNoIGRpbWVuc2lvbiBvZiB0aGUgd2luZG93DQogICAgICAgIHNzIC0gYW4gaW50IChhIGlzIDFEKSBvciB0dXBsZSAoYSBpcyAyRCBvciBncmVhdGVyKSByZXByZXNlbnRpbmcgdGhlDQogICAgICAgICAgICAgYW1vdW50IHRvIHNsaWRlIHRoZSB3aW5kb3cgaW4gZWFjaCBkaW1lbnNpb24uIElmIG5vdCBzcGVjaWZpZWQsIGl0DQogICAgICAgICAgICAgZGVmYXVsdHMgdG8gd3MuDQogICAgICAgIGZsYXR0ZW4gLSBpZiBUcnVlLCBhbGwgc2xpY2VzIGFyZSBmbGF0dGVuZWQsIG90aGVyd2lzZSwgdGhlcmUgaXMgYW4NCiAgICAgICAgICAgICAgICAgIGV4dHJhIGRpbWVuc2lvbiBmb3IgZWFjaCBkaW1lbnNpb24gb2YgdGhlIGlucHV0Lg0KDQogICAgUmV0dXJucw0KICAgICAgICBhbiBhcnJheSBjb250YWluaW5nIGVhY2ggbi1kaW1lbnNpb25hbCB3aW5kb3cgZnJvbSBhDQogICAgJycnDQoNCiAgICBpZiBOb25lIGlzIHNzOg0KICAgICAgICAjIHNzIHdhcyBub3QgcHJvdmlkZWQuIHRoZSB3aW5kb3dzIHdpbGwgbm90IG92ZXJsYXAgaW4gYW55IGRpcmVjdGlvbi4NCiAgICAgICAgc3MgPSB3cw0KICAgIHdzID0gbm9ybV9zaGFwZSh3cykNCiAgICBzcyA9IG5vcm1fc2hhcGUoc3MpDQoNCiAgICAjIGNvbnZlcnQgd3MsIHNzLCBhbmQgYS5zaGFwZSB0byBudW1weSBhcnJheXMgc28gdGhhdCB3ZSBjYW4gZG8gbWF0aCBpbiBldmVyeQ0KICAgICMgZGltZW5zaW9uIGF0IG9uY2UuDQogICAgd3MgPSBucC5hcnJheSh3cykNCiAgICBzcyA9IG5wLmFycmF5KHNzKQ0KICAgIHNoYXBlID0gbnAuYXJyYXkoYS5zaGFwZSkNCg0KDQogICAgIyBlbnN1cmUgdGhhdCB3cywgc3MsIGFuZCBhLnNoYXBlIGFsbCBoYXZlIHRoZSBzYW1lIG51bWJlciBvZiBkaW1lbnNpb25zDQogICAgbHMgPSBbbGVuKHNoYXBlKSxsZW4od3MpLGxlbihzcyldDQogICAgaWYgMSAhPSBsZW4oc2V0KGxzKSk6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoXA0KICAgICAgICAnYS5zaGFwZSwgd3MgYW5kIHNzIG11c3QgYWxsIGhhdmUgdGhlIHNhbWUgbGVuZ3RoLiBUaGV5IHdlcmUgJXMnICUgc3RyKGxzKSkNCg0KICAgICMgZW5zdXJlIHRoYXQgd3MgaXMgc21hbGxlciB0aGFuIGEgaW4gZXZlcnkgZGltZW5zaW9uDQogICAgaWYgbnAuYW55KHdzID4gc2hhcGUpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKFwNCiAgICAgICAgJ3dzIGNhbm5vdCBiZSBsYXJnZXIgdGhhbiBhIGluIGFueSBkaW1lbnNpb24uXA0KIGEuc2hhcGUgd2FzICVzIGFuZCB3cyB3YXMgJXMnICUgKHN0cihhLnNoYXBlKSxzdHIod3MpKSkNCg0KICAgICMgaG93IG1hbnkgc2xpY2VzIHdpbGwgdGhlcmUgYmUgaW4gZWFjaCBkaW1lbnNpb24/DQogICAgbmV3c2hhcGUgPSBub3JtX3NoYXBlKCgoc2hhcGUgLSB3cykgLy8gc3MpICsgMSkNCiAgICAjIHRoZSBzaGFwZSBvZiB0aGUgc3RyaWRlZCBhcnJheSB3aWxsIGJlIHRoZSBudW1iZXIgb2Ygc2xpY2VzIGluIGVhY2ggZGltZW5zaW9uDQogICAgIyBwbHVzIHRoZSBzaGFwZSBvZiB0aGUgd2luZG93ICh0dXBsZSBhZGRpdGlvbikNCiAgICBuZXdzaGFwZSArPSBub3JtX3NoYXBlKHdzKQ0KICAgICMgdGhlIHN0cmlkZXMgdHVwbGUgd2lsbCBiZSB0aGUgYXJyYXkncyBzdHJpZGVzIG11bHRpcGxpZWQgYnkgc3RlcCBzaXplLCBwbHVzDQogICAgIyB0aGUgYXJyYXkncyBzdHJpZGVzICh0dXBsZSBhZGRpdGlvbikNCiAgICBuZXdzdHJpZGVzID0gbm9ybV9zaGFwZShucC5hcnJheShhLnN0cmlkZXMpICogc3MpICsgYS5zdHJpZGVzDQogICAgc3RyaWRlZCA9IGFzdChhLHNoYXBlID0gbmV3c2hhcGUsc3RyaWRlcyA9IG5ld3N0cmlkZXMpDQogICAgaWYgbm90IGZsYXR0ZW46DQogICAgICAgIHJldHVybiBzdHJpZGVkDQoNCiAgICAjIENvbGxhcHNlIHN0cmlkZWQgc28gdGhhdCBpdCBoYXMgb25lIG1vcmUgZGltZW5zaW9uIHRoYW4gdGhlIHdpbmRvdy4gIEkuZS4sDQogICAgIyB0aGUgbmV3IGFycmF5IGlzIGEgZmxhdCBsaXN0IG9mIHNsaWNlcy4NCiAgICBtZWF0ID0gbGVuKHdzKSBpZiB3cy5zaGFwZSBlbHNlIDANCiAgICBmaXJzdGRpbSA9IChucC5wcm9kKG5ld3NoYXBlWzotbWVhdF0pLCkgaWYgd3Muc2hhcGUgZWxzZSAoKQ0KICAgIGRpbSA9IGZpcnN0ZGltICsgKG5ld3NoYXBlWy1tZWF0Ol0pDQogICAgcmV0dXJuIHN0cmlkZWQucmVzaGFwZShkaW0pDQo="}, "code/train.py": {"sha256": "4e52c09204dc4bb91ff7eb8cd5f0010cdf272760e9ae95ce00947b3e176806e7", "b64": "ZnJvbSB1dGlscyBpbXBvcnQgKg0KaW1wb3J0IHRxZG0NCmFjY19hbGwsIHJlc3VsdCA9IFtdLCBbXQ0KaW1wb3J0IHRvcmNoDQoNCg0KZGVmIHRyYWluX2F1eGkoYXJncywgc291cmNlX2xvYWRlcnMsIERFVklDRSwgbW9kZWwsIG9wdGltaXplciwgZSwgZmFsc2Vfb3B0aW1pemVyKToNCiAgICBtb2RlbC50cmFpbigpDQogICAgdHJhaW5fbG9zcyA9IDANCiAgICBlcG9jaF9jbGFzc195X2xvc3MgPSAwDQogICAgdG90YWwgPSAwDQogICAgZm9yIHNvdXJjZV9sb2FkZXIgaW4gc291cmNlX2xvYWRlcnM6DQogICAgICAgIGZvciBiYXRjaF9pZHgsICh4LCB5LCBkKSBpbiBlbnVtZXJhdGUoc291cmNlX2xvYWRlcik6DQogICAgICAgICAgICB4LCB5LCBkID0geC50byhERVZJQ0UpLCB5LnRvKERFVklDRSksIGQudG8oREVWSUNFKQ0KDQogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkNCiAgICAgICAgICAgIGZhbHNlX29wdGltaXplci56ZXJvX2dyYWQoKQ0KDQogICAgICAgICAgICBsb3NzX29yaWdpbiwgY2xhc3NfeV9sb3NzID0gbW9kZWwubG9zc19mdW5jdGlvbihkLCB4LCB5KQ0KICAgICAgICAgICAgbG9zc19vcmlnaW4gPSBsb3NzX29yaWdpbg0KDQogICAgICAgICAgICBsb3NzX2ZhbHNlID0gbW9kZWwubG9zc19mdW5jdGlvbl9mYWxzZShhcmdzLCBkLCB4LCB5KQ0KDQogICAgICAgICAgICBsb3NzX29yaWdpbi5iYWNrd2FyZCgpDQogICAgICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpDQogICAgICAgICAgICBsb3NzX2ZhbHNlLmJhY2t3YXJkKCkNCiAgICAgICAgICAgIGZhbHNlX29wdGltaXplci5zdGVwKCkNCg0KICAgICAgICAgICAgdHJhaW5fbG9zcyArPSBsb3NzX29yaWdpbg0KICAgICAgICAgICAgZXBvY2hfY2xhc3NfeV9sb3NzICs9IGNsYXNzX3lfbG9zcw0KICAgICAgICAgICAgdG90YWwgKz0geS5zaXplKDApDQoNCiAgICB0cmFpbl9sb3NzIC89IHRvdGFsDQogICAgZXBvY2hfY2xhc3NfeV9sb3NzIC89IHRvdGFsDQoNCiAgICByZXR1cm4gdHJhaW5fbG9zcywgZXBvY2hfY2xhc3NfeV9sb3NzDQoNCg0KZGVmIGdldF9hY2N1cmFjeShzb3VyY2VfbG9hZGVycywgREVWSUNFLCBtb2RlbCwgY2xhc3NpZmllcl9mbiwgYmF0Y2hfc2l6ZSk6DQogICAgbW9kZWwuZXZhbCgpDQogICAgIiIiDQogICAgY29tcHV0ZSB0aGUgYWNjdXJhY3kgb3ZlciB0aGUgc3VwZXJ2aXNlZCB0cmFpbmluZyBzZXQgb3IgdGhlIHRlc3Rpbmcgc2V0DQogICAgIiIiDQogICAgcHJlZGljdGlvbnNfZCwgYWN0dWFsc19kLCBwcmVkaWN0aW9uc195LCBhY3R1YWxzX3kgPSBbXSwgW10sIFtdLCBbXQ0KICAgIHByZWRpY3Rpb25zX2RfZmFsc2UsIHByZWRpY3Rpb25zX3lfZmFsc2UgPSBbXSwgW10NCg0KICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICBmb3Igc291cmNlX2xvYWRlciBpbiBzb3VyY2VfbG9hZGVyczoNCiAgICAgICAgICAgIGZvciAoeHMsIHlzLCBkcykgaW4gc291cmNlX2xvYWRlcjoNCg0KICAgICAgICAgICAgICAgIHhzLCB5cywgZHMgPSB4cy50byhERVZJQ0UpLCB5cy50byhERVZJQ0UpLCBkcy50byhERVZJQ0UpDQoNCiAgICAgICAgICAgICAgICAjIHVzZSBjbGFzc2lmaWNhdGlvbiBmdW5jdGlvbiB0byBjb21wdXRlIGFsbCBwcmVkaWN0aW9ucyBmb3IgZWFjaCBiYXRjaA0KICAgICAgICAgICAgICAgIHByZWRfZCwgcHJlZF95LCBwcmVkX2RfZmFsc2UsIHByZWRfeV9mYWxzZSA9IGNsYXNzaWZpZXJfZm4oeHMpDQogICAgICAgICAgICAgICAgcHJlZGljdGlvbnNfZC5hcHBlbmQocHJlZF9kKQ0KICAgICAgICAgICAgICAgIHByZWRpY3Rpb25zX2RfZmFsc2UuYXBwZW5kKHByZWRfZF9mYWxzZSkNCiAgICAgICAgICAgICAgICBhY3R1YWxzX2QuYXBwZW5kKGRzKQ0KDQogICAgICAgICAgICAgICAgcHJlZGljdGlvbnNfeS5hcHBlbmQocHJlZF95KQ0KICAgICAgICAgICAgICAgIHByZWRpY3Rpb25zX3lfZmFsc2UuYXBwZW5kKHByZWRfeV9mYWxzZSkNCiAgICAgICAgICAgICAgICBhY3R1YWxzX3kuYXBwZW5kKHlzKQ0KDQogICAgICAgICMgY29tcHV0ZSB0aGUgbnVtYmVyIG9mIGFjY3VyYXRlIHByZWRpY3Rpb25zDQogICAgICAgIGFjY3VyYXRlX3ByZWRzX2QgPSAwDQogICAgICAgIGFjY3VyYXRlX3ByZWRzX2RfZmFsc2UgPSAwDQogICAgICAgIGZvciBwcmVkLCBhY3QgaW4gemlwKHByZWRpY3Rpb25zX2QsIGFjdHVhbHNfZCk6DQogICAgICAgICAgICBfLCBudW1fcHJlZCA9IHByZWQubWF4KDEpDQogICAgICAgICAgICB2ID0gdG9yY2guc3VtKG51bV9wcmVkID09IGFjdCkNCiAgICAgICAgICAgIGFjY3VyYXRlX3ByZWRzX2QgKz0gdg0KDQogICAgICAgIGFjY3VyYWN5X2QgPSAoYWNjdXJhdGVfcHJlZHNfZCAqIDEuMCkgLyAobGVuKHByZWRpY3Rpb25zX2QpICogYmF0Y2hfc2l6ZSkNCg0KICAgICAgICBmb3IgcHJlZCwgYWN0IGluIHppcChwcmVkaWN0aW9uc19kX2ZhbHNlLCBhY3R1YWxzX2QpOg0KICAgICAgICAgICAgXywgbnVtX3ByZWQgPSBwcmVkLm1heCgxKQ0KICAgICAgICAgICAgdiA9IHRvcmNoLnN1bShudW1fcHJlZCA9PSBhY3QpDQogICAgICAgICAgICBhY2N1cmF0ZV9wcmVkc19kX2ZhbHNlICs9IHYNCg0KICAgICAgICAjIGNhbGN1bGF0ZSB0aGUgYWNjdXJhY3kgYmV0d2VlbiAwIGFuZCAxDQogICAgICAgIGFjY3VyYWN5X2RfZmFsc2UgPSAoYWNjdXJhdGVfcHJlZHNfZF9mYWxzZSAqIDEuMCkgLyAobGVuKHByZWRpY3Rpb25zX2RfZmFsc2UpICogYmF0Y2hfc2l6ZSkNCg0KICAgICAgICAjIGNvbXB1dGUgdGhlIG51bWJlciBvZiBhY2N1cmF0ZSBwcmVkaWN0aW9ucw0KICAgICAgICBhY2N1cmF0ZV9wcmVkc195ID0gMA0KICAgICAgICBhY2N1cmF0ZV9wcmVkc195X2ZhbHNlID0gMA0KDQogICAgICAgIGZvciBwcmVkLCBhY3QgaW4gemlwKHByZWRpY3Rpb25zX3ksIGFjdHVhbHNfeSk6DQogICAgICAgICAgICBfLCBudW1fcHJlZCA9IHByZWQubWF4KDEpDQogICAgICAgICAgICB2ID0gdG9yY2guc3VtKG51bV9wcmVkID09IGFjdCkNCiAgICAgICAgICAgIGFjY3VyYXRlX3ByZWRzX3kgKz0gdg0KDQogICAgICAgIGFjY3VyYWN5X3kgPSAoYWNjdXJhdGVfcHJlZHNfeSAqIDEuMCkgLyAobGVuKHByZWRpY3Rpb25zX3kpICogYmF0Y2hfc2l6ZSkNCg0KDQogICAgICAgIGZvciBwcmVkLCBhY3QgaW4gemlwKHByZWRpY3Rpb25zX3lfZmFsc2UsIGFjdHVhbHNfeSk6DQogICAgICAgICAgICBfLCBudW1fcHJlZCA9IHByZWQubWF4KDEpDQogICAgICAgICAgICB2ID0gdG9yY2guc3VtKG51bV9wcmVkID09IGFjdCkNCiAgICAgICAgICAgIGFjY3VyYXRlX3ByZWRzX3lfZmFsc2UgKz0gdg0KICAgICAgICAjIGNhbGN1bGF0ZSB0aGUgYWNjdXJhY3kgYmV0d2VlbiAwIGFuZCAxDQogICAgICAgIGFjY3VyYWN5X3lfZmFsc2UgPSAoYWNjdXJhdGVfcHJlZHNfeV9mYWxzZSAqIDEuMCkgLyAobGVuKHByZWRpY3Rpb25zX3lfZmFsc2UpICogYmF0Y2hfc2l6ZSkNCg0KICAgICAgICByZXR1cm4gYWNjdXJhY3lfZCwgYWNjdXJhY3lfeSwgYWNjdXJhY3lfZF9mYWxzZSwgYWNjdXJhY3lfeV9mYWxzZQ0KDQoNCg0KZGVmIHRyYWluX0dJTEUobW9kZWwsIERFVklDRSwgb3B0aW1pemVyLCBzb3VyY2VfbG9hZGVycywgdGFyZ2V0X2xvYWRlciwgcmVzdWx0X2NzdiwgcmVzdWx0X3R4dCwgYXJncyk6DQogICAgYWNjX2FsbCwgcmVzdWx0ID0gW10sIFtdDQogICAgYmVzdF9hY2MsIGJlc3RfaXRlciA9IDAuMCwgMA0KDQogICAgZmFsc2Vfb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbShtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPTFlLTMpDQoNCiAgICBmb3IgZSBpbiByYW5nZShhcmdzLm5fZXBvY2gpOg0KICAgICAgICBhdmdfZXBvY2hfbG9zcywgYXZnX2Vwb2NoX2NsYXNzX3lfbG9zcyA9IHRyYWluX2F1eGkoYXJncywgc291cmNlX2xvYWRlcnMsIERFVklDRSwgbW9kZWwsIG9wdGltaXplciwgZSwgZmFsc2Vfb3B0aW1pemVyKQ0KICAgICAgICB0cWRtLnRxZG0ud3JpdGUoJ0Vwb2NoOiBbe30ve31dLCBhdmcgbG9zczogezouMmZ9LCB5IGxvc3M6IHs6LjJmfSUnLmZvcm1hdChlICsgMSwgYXJncy5uX2Vwb2NoLCBhdmdfZXBvY2hfbG9zcywgYXZnX2Vwb2NoX2NsYXNzX3lfbG9zcykpDQogICAgICAgIHRyYWluX2FjY19kLCB0cmFpbl9hY2NfeSwgdHJhaW5fYWNjX2RfZmFsc2UsIHRyYWluX2FjY195X2ZhbHNlID0gZ2V0X2FjY3VyYWN5KHNvdXJjZV9sb2FkZXJzLCBERVZJQ0UsIG1vZGVsLCBtb2RlbC5jbGFzc2lmaWVyLCBhcmdzLmJhdGNoX3NpemUpDQogICAgICAgIHRlc3RfYWNjX2QsIHRlc3RfYWNjX3ksIHRlc3RfYWNjX2RfZmFsc2UsIHRlc3RfYWNjX3lfZmFsc2UgPSBnZXRfYWNjdXJhY3koW3RhcmdldF9sb2FkZXJdLCBERVZJQ0UsIG1vZGVsLCBtb2RlbC5jbGFzc2lmaWVyLCBhcmdzLmJhdGNoX3NpemUpDQogICAgICAgIHRxZG0udHFkbS53cml0ZSgnRXBvY2g6W3t9L3t9XSwgdHJhaW4gYWNjIGQ6ezouMmZ9IHk6ezouMmZ9IHwgZF9mYWxzZTp7Oi4yZn0geV9mYWxzZTp7Oi4yZn0nLmZvcm1hdChlKzEsIGFyZ3Mubl9lcG9jaCwgdHJhaW5fYWNjX2QsIHRyYWluX2FjY195LCB0cmFpbl9hY2NfZF9mYWxzZSwgdHJhaW5fYWNjX3lfZmFsc2UpKQ0KICAgICAgICB0cWRtLnRxZG0ud3JpdGUoJ0Vwb2NoOlt7fS97fV0sIFRFU1QgYWNjIGQ6ezouMmZ9IHk6ezouMmZ9IHwgZF9mYWxzZTp7Oi4yZn0geV9mYWxzZTp7Oi4yZn0nLmZvcm1hdChlKzEsIGFyZ3Mubl9lcG9jaCwgdGVzdF9hY2NfZCwgdGVzdF9hY2NfeSwgdGVzdF9hY2NfZF9mYWxzZSwgdGVzdF9hY2NfeV9mYWxzZSkpDQoNCiAgICAgICAgIyBHSUxFX0VWQUxVQVRJT05fQ0hFQ0tQT0lOVF9QQVRDSA0KICAgICAgICAjIFNhdmUgd2VpZ2h0cyB3aGVuZXZlciB0aGUgb3JpZ2luYWwgdGFyZ2V0LWFjY3VyYWN5IGNyaXRlcmlvbiBpbXByb3Zlcy4NCiAgICAgICAgaWYgdGVzdF9hY2NfeS5pdGVtKCkgPiBiZXN0X2FjYzoNCiAgICAgICAgICAgIHRvcmNoLnNhdmUoDQogICAgICAgICAgICAgICAgbW9kZWwuc3RhdGVfZGljdCgpLA0KICAgICAgICAgICAgICAgIG9zLnBhdGguam9pbihhcmdzLmV2YWxfY2hlY2twb2ludF9kaXIsICdiZXN0X21vZGVsLnB0JykNCiAgICAgICAgICAgICkNCg0KICAgICAgICBhY2NfYWxsLmFwcGVuZCh0ZXN0X2FjY195KQ0KICAgICAgICBiZXN0X2FjYyA9IG1heChhY2NfYWxsKS5pdGVtKCkNCiAgICAgICAgYmVzdF9pdGVyID0gYWNjX2FsbC5pbmRleChiZXN0X2FjYykgKyAxDQogICAgICAgIHJlc3VsdC5hcHBlbmQoW3RyYWluX2FjY195LCB0ZXN0X2FjY195LCBiZXN0X2FjYywgYmVzdF9pdGVyLCB0cmFpbl9hY2NfZF0pDQogICAgICAgIHJlc3VsdF9ucCA9IG5wLmFycmF5KFsNCiAgICAgICAgICAgIFsNCiAgICAgICAgICAgICAgICB2YWx1ZS5kZXRhY2goKS5jcHUoKS5pdGVtKCkNCiAgICAgICAgICAgICAgICBpZiB0b3JjaC5pc190ZW5zb3IodmFsdWUpIGVsc2UgdmFsdWUNCiAgICAgICAgICAgICAgICBmb3IgdmFsdWUgaW4gcm93DQogICAgICAgICAgICBdDQogICAgICAgICAgICBmb3Igcm93IGluIHJlc3VsdA0KICAgICAgICBdLCBkdHlwZT1mbG9hdCkNCiAgICAgICAgbnAuc2F2ZXR4dChyZXN1bHRfY3N2LCByZXN1bHRfbnAsIGZtdD0nJS4yZicsIGRlbGltaXRlcj0nLCcpDQogICAgbG9nX3Jlc3VsdF9jb25jaXNlKGFyZ3MsIHJlc3VsdF90eHQsIFtiZXN0X2FjYywgYmVzdF9pdGVyXSkNCiAgICBwbG90KHJlc3VsdF9jc3YpDQoNCg0KZGVmIHRyYWluKG1vZGVsLCBERVZJQ0UsIG9wdGltaXplciwgc291cmNlX2xvYWRlcnMsIHRhcmdldF9sb2FkZXIsIHJlc3VsdF9jc3YsIHJlc3VsdF90eHQsIGFyZ3MpOg0KICAgIGlmIGFyZ3Mubm93X21vZGVsX25hbWUgaW4gWydHSUxFJ106DQogICAgICAgIHJldHVybiB0cmFpbl9HSUxFKG1vZGVsLCBERVZJQ0UsIG9wdGltaXplciwgc291cmNlX2xvYWRlcnMsIHRhcmdldF9sb2FkZXIsIHJlc3VsdF9jc3YsIHJlc3VsdF90eHQsIGFyZ3MpDQogICAgZWxzZToNCiAgICAgICAgcGFzcw=="}, "code/utils.py": {"sha256": "bf30eb45ea47e6c95b555365b716b416831de78db812215527bf2342f9dbac53", "b64": "aW1wb3J0IHRvcmNoDQppbXBvcnQgb3MNCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBmMV9zY29yZQ0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0DQpmcm9tIHNsaWRpbmdfd2luZG93IGltcG9ydCBzbGlkaW5nX3dpbmRvdw0KDQpkZWYgZ2V0X3NhbXBsZV93ZWlnaHRzKHksIHdlaWdodHMpOg0KICAgICcnJw0KICAgIHRvIGFzc2lnbiB3ZWlnaHRzIHRvIGVhY2ggc2FtcGxlDQogICAgJycnDQogICAgbGFiZWxfdW5pcXVlID0gbnAudW5pcXVlKHkpDQogICAgc2FtcGxlX3dlaWdodHMgPSBbXQ0KICAgIGZvciB2YWwgaW4geToNCiAgICAgICAgaWR4ID0gbnAud2hlcmUobGFiZWxfdW5pcXVlID09IHZhbCkNCiAgICAgICAgc2FtcGxlX3dlaWdodHMuYXBwZW5kKHdlaWdodHNbaWR4XSkNCiAgICByZXR1cm4gc2FtcGxlX3dlaWdodHMNCg0KDQpkZWYgb3BwX3NsaWRpbmdfd2luZG93X3dfZChkYXRhX3gsIGRhdGFfeSwgZCwgd3MsIHNzKTogIyB3aW5kb3cgc2l6ZSwgc3RlcCBzaXplDQogICAgZGF0YV94ID0gc2xpZGluZ193aW5kb3coZGF0YV94LCh3cyxkYXRhX3guc2hhcGVbMV0pLChzcywxKSkNCiAgICBkYXRhX3kgPSBucC5hc2FycmF5KFtbaVstMV1dIGZvciBpIGluIHNsaWRpbmdfd2luZG93KGRhdGFfeSx3cyxzcyldKQ0KICAgIGRhdGFfZCA9IG5wLmFzYXJyYXkoW1tpWy0xXV0gZm9yIGkgaW4gc2xpZGluZ193aW5kb3coZCwgd3MsIHNzKV0pDQogICAgcmV0dXJuIGRhdGFfeC5hc3R5cGUobnAuZmxvYXQzMiksIGRhdGFfeS5yZXNoYXBlKGxlbihkYXRhX3kpKS5hc3R5cGUobnAudWludDgpLCBkYXRhX2QucmVzaGFwZShsZW4oZGF0YV9kKSkuYXN0eXBlKG5wLnVpbnQ4KQ0KDQoNCmRlZiBwbG90KHJlc3VsdF9jc3YpOg0KICAgIGRhdGEgPSBucC5sb2FkdHh0KHJlc3VsdF9jc3YsIGRlbGltaXRlcj0nLCcpDQogICAgcGx0LmZpZ3VyZSgpDQogICAgcGx0LnBsb3QocmFuZ2UoMSwgbGVuKGRhdGFbOiwgMF0pICsgMSksIGRhdGFbOiwgMF0sIGNvbG9yPSdibHVlJywgbGFiZWw9J3RyYWluJykNCiAgICBwbHQucGxvdChyYW5nZSgxLCBsZW4oZGF0YVs6LCAxXSkgKyAxKSwgZGF0YVs6LCAxXSwgY29sb3I9J3JlZCcsIGxhYmVsPSd0ZXN0JykNCiAgICBwbHQubGVnZW5kKCkNCiAgICBwbHQueGxhYmVsKCdFcG9jaCcsIGZvbnRzaXplPTE0KQ0KICAgIHBsdC55bGFiZWwoJ0FjY3VyYWN5ICglKScsIGZvbnRzaXplPTE0KQ0KICAgIHBsdC50aXRsZSgnVHJhaW5pbmcgYW5kIFRlc3QgQWNjdXJhY3knLCBmb250c2l6ZT0yMCkNCiAgICBwbHQuc2F2ZWZpZyhyZXN1bHRfY3N2WzotNF0rJ19maWd1cmUucG5nJykNCg0KDQpkZWYgbG9nX3Jlc3VsdChhcmdzLCByZXN1bHRfdHh0LCBiZXN0KToNCiAgICBsb2dfYXJncyhhcmdzLCByZXN1bHRfdHh0KQ0KICAgIHdpdGggb3BlbihyZXN1bHRfdHh0LCAnYScpIGFzIGY6DQogICAgICAgIGYud3JpdGUoJ2VfYWNjOiAnK3N0cihiZXN0WzBdKVs6Nl0NCiAgICAgICAgICAgICAgICArICdcdCBlX21pRjogJytzdHIoYmVzdFsxXSlbOjZdICsgJ1x0IGVfbWFGOiAnK3N0cihiZXN0WzJdKVs6Nl0NCiAgICAgICAgICAgICAgICArICdcdCBmX2FjYzogJytzdHIoYmVzdFszXSlbOjZdICsgJ1x0IGZfbWlGOiAnK3N0cihiZXN0WzRdKVs6Nl0NCiAgICAgICAgICAgICAgICArICdcdCBmX21hRjogJytzdHIoYmVzdFs1XSlbOjZdDQogICAgICAgICAgICAgICAgKyAnXHQgYmVzdF9pdGVyOiAnK3N0cihiZXN0WzZdKSArICdcblxuJykNCg0KZGVmIGxvZ19yZXN1bHRfY29uY2lzZShhcmdzLCByZXN1bHRfdHh0LCBiZXN0KToNCiAgICBsb2dfYXJncyhhcmdzLCByZXN1bHRfdHh0KQ0KICAgIHdpdGggb3BlbihyZXN1bHRfdHh0LCAnYScpIGFzIGY6DQogICAgICAgIGYud3JpdGUoJ2VfYWNjOiAnK3N0cihiZXN0WzBdKVs6Nl0gKyAnXHQgYmVzdF9pdGVyOiAnK3N0cihiZXN0WzFdKSArICdcblxuJykNCiAgICBjb25jaXNlX3R4dCA9IHJlc3VsdF90eHRbOi00XSArICdfcHVyZS50eHQnDQogICAgd2l0aCBvcGVuKGNvbmNpc2VfdHh0LCAnYScpIGFzIGZmOg0KICAgICAgICBmZi53cml0ZShzdHIoYmVzdFswXSlbOjZdICsgJ1xuJykNCg0KZGVmIHByaW50X2FyZ3MoYXJncyk6DQogICAgb3V0X3N0ciA9ICcnDQogICAgZm9yIGFyZ3VtZW50LCB2YWx1ZSBpbiBzb3J0ZWQodmFycyhhcmdzKS5pdGVtcygpKToNCiAgICAgICAgb3V0X3N0ciArPSBzdHIoYXJndW1lbnQpICsgJzonICsgc3RyKHZhbHVlKSArICdfJw0KICAgIHJldHVybiBvdXRfc3RyDQoNCmRlZiBsb2dfYXJncyhhcmdzLCByZXN1bHRfdHh0KToNCiAgICB3aXRoIG9wZW4ocmVzdWx0X3R4dCwgJ2EnKSBhcyBmOg0KICAgICAgICBmLndyaXRlKCdcblNldHRpbmc6IE1vZGVsOiAnICsgYXJncy5ub3dfbW9kZWxfbmFtZSArICcgVGFyZ2V0OiAnICsgYXJncy50YXJnZXRfZG9tYWluICsgJ1xuJykNCiAgICAgICAgZm9yIGFyZ3VtZW50LCB2YWx1ZSBpbiBzb3J0ZWQodmFycyhhcmdzKS5pdGVtcygpKToNCiAgICAgICAgICAgIGYud3JpdGUoJ3t9Ont9ICcuZm9ybWF0KGFyZ3VtZW50LCB2YWx1ZSkpDQoNCmRlZiBzZXRfbmFtZShhcmdzKToNCiAgICBjdXJfcGF0aCA9ICdyZXN1bHRzLycgKyBhcmdzLmRhdGFzZXQgKyAnLycgKyBhcmdzLm5vd19tb2RlbF9uYW1lICsgJy8nICsgYXJncy50YXJnZXRfZG9tYWluICsgJy8nDQogICAgZGlyX25hbWUgPSAncmVzdWx0cy8nICsgYXJncy5kYXRhc2V0ICsgJy8nICsgYXJncy5ub3dfbW9kZWxfbmFtZSArICcvJyArIGFyZ3MudGFyZ2V0X2RvbWFpbg0KICAgIHJlc3VsdF9jc3YgPSBjdXJfcGF0aCArIHN0cihhcmdzLm5fZXBvY2gpICsgJ18nICsgc3RyKGFyZ3MuYmF0Y2hfc2l6ZSkgKyAnXycgKyBhcmdzLm5vd19tb2RlbF9uYW1lICsnLmNzdicNCg0KICAgIHJlc3VsdF90eHQgPSBjdXJfcGF0aCArICdiZXN0X3Jlc3VsdF8nK2FyZ3MuZGF0YXNldCArICcudHh0Jw0KICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhkaXJfbmFtZSk6DQogICAgICAgIG9zLm1ha2VkaXJzKGRpcl9uYW1lKQ0KICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhjdXJfcGF0aCk6DQogICAgICAgIG9zLm1ha2VkaXJzKGN1cl9wYXRoWzotMV0pDQogICAgaWYgbm90IG9zLnBhdGguaXNmaWxlKHJlc3VsdF9jc3YpOg0KICAgICAgICB3aXRoIG9wZW4ocmVzdWx0X2NzdiwgJ3cnKSBhcyBteV9lbXB0eV9jc3Y6DQogICAgICAgICAgICBwYXNzDQogICAgcmV0dXJuIHJlc3VsdF9jc3YsIHJlc3VsdF90eHQsIGRpcl9uYW1lDQoNCg0KZGVmIGdldF9zY2FsZV9tYXRyaXgoTSwgTik6DQogICAgczEgPSB0b3JjaC5vbmVzKChOLCAxKSkgKiAxLjAgLyBODQogICAgczIgPSB0b3JjaC5vbmVzKChNLCAxKSkgKiAtMS4wIC8gTQ0KICAgIHJldHVybiB0b3JjaC5jYXQoKHMxLCBzMiksIDApDQoNCmRlZiBtbWRfY3VzdG9ybShzYW1wbGUsIGRlY29kZWQsIHNpZ21hPVsxXSk6ICMgMC4xLCAxLCAxMA0KICAgIFggPSB0b3JjaC5jYXQoKGRlY29kZWQsIHNhbXBsZSksIDApDQogICAgWFggPSB0b3JjaC5tYXRtdWwoWCwgWC50KCkpDQogICAgWDIgPSB0b3JjaC5zdW0oWCAqIFgsIDEsIGtlZXBkaW09VHJ1ZSkNCiAgICBleHAgPSBYWCAtIDAuNSAqIFgyIC0gMC41ICogWDIudCgpDQoNCiAgICBNID0gZGVjb2RlZC5zaXplKClbMF0NCiAgICBOID0gc2FtcGxlLnNpemUoKVswXQ0KICAgIHMgPSBnZXRfc2NhbGVfbWF0cml4KE0sIE4pDQogICAgUyA9IHRvcmNoLm1hdG11bChzLCBzLnQoKSkNCg0KICAgIGxvc3MgPSAwDQogICAgZm9yIHYgaW4gc2lnbWE6DQogICAgICAgIGtlcm5lbF92YWwgPSB0b3JjaC5leHAoZXhwIC8gdikNCiAgICAgICAga2VybmVsX3ZhbCA9IGtlcm5lbF92YWwuY3B1KCkNCiAgICAgICAgbG9zcyArPSB0b3JjaC5zdW0oUyAqIGtlcm5lbF92YWwpDQoNCiAgICBsb3NzX21tZCA9IHRvcmNoLnNxcnQobG9zcykNCiAgICByZXR1cm4gbG9zc19tbWQNCg0KDQpkZWYgbWVhc3VyZV9ldmVudF9mcmFtZShwcmVkaWN0ZWRfbGFiZWxfc2VnbWVudCwgbGVuZ3Roc192YXJ5aW5nX3NlZ21lbnQsIHRydWVfbGFiZWxfc2VnbWVudCk6DQogICAgIiIiDQogICAgdGhpcyBmdW5jdGlvbiByZXR1cm5zIHRoZSBjb3JyZWN0IG1lYXN1cmVtZW50cyAoYm90aCBmcmFtZS0gYW5kIGV2ZW50LWxldmVsKSBmb3IgY2h1bmstYmFzZWQgcHJlZGljdGlvbiBvbiBhY3Rpdml0eQ0KICAgIG5vdGljZSB0aGF0ICdtYWNybycgb3B0aW9uIGluIHNrbGVhcm4gZG9lcyBub3QgcmV0dXJuIHRoZSBkZXNpcmVkIHdlaWdodGVkIG1hRjsgdGhlcmVmb3JlICd3ZWlnaHRlZCcgb3B0aW9uIGlzIHVzZWQgaW5zdGVhZA0KICAgICIiIg0KICAgIGV2ZW50X2FjYywgZXZlbnRfbWlGLCBldmVudF9tYUYsIGZyYW1lX2FjYywgZnJhbWVfbWlGLCBmcmFtZV9tYUYgPSAwLjAsIDAuMCwgMC4wLCAwLjAsIDAuMCwgMC4wDQoNCiAgICBuX2V2ZW50ID0gdHJ1ZV9sYWJlbF9zZWdtZW50LnNpemUoMCkNCiAgICBldmVudF9hY2MgPSAoKHByZWRpY3RlZF9sYWJlbF9zZWdtZW50ID09IHRydWVfbGFiZWxfc2VnbWVudCkuc3VtKCkpLmRvdWJsZSgpICogMTAwIC8gbl9ldmVudA0KICAgIGV2ZW50X21pRiA9IGYxX3Njb3JlKHRydWVfbGFiZWxfc2VnbWVudCwgcHJlZGljdGVkX2xhYmVsX3NlZ21lbnQsIGF2ZXJhZ2U9J21pY3JvJykgKiAxMDANCiAgICBldmVudF9tYUYgPSBmMV9zY29yZSh0cnVlX2xhYmVsX3NlZ21lbnQsIHByZWRpY3RlZF9sYWJlbF9zZWdtZW50LCBhdmVyYWdlPSd3ZWlnaHRlZCcpICogMTAwDQoNCiAgICAjIGNyZWF0ZSBmcmFtZS1iYXNlZCB2ZWN0b3JzDQogICAgbl9mcmFtZSA9IHN1bShsZW5ndGhzX3Zhcnlpbmdfc2VnbWVudCkNCiAgICBwcmVkaWN0ZWRfbGFiZWxfZnJhbWUsIHRydWVfbGFiZWxfZnJhbWUgPSB0b3JjaC5Mb25nVGVuc29yKCksIHRvcmNoLkxvbmdUZW5zb3IoKQ0KICAgIHByZWRpY3RlZF9sYWJlbF9mcmFtZSA9IHRvcmNoLmNhdChbdG9yY2guY2F0KChwcmVkaWN0ZWRfbGFiZWxfZnJhbWUsIHByZWRpY3RlZF9sYWJlbF9zZWdtZW50W2ldLnJlcGVhdChsZW5ndGhzX3Zhcnlpbmdfc2VnbWVudFtpXSwxKSksIGRpbT0wKSBmb3IgaSBpbiByYW5nZShuX2V2ZW50KV0pDQogICAgYXNzZXJ0IHByZWRpY3RlZF9sYWJlbF9mcmFtZS5zaGFwZVswXSA9PSBuX2ZyYW1lDQoNCiAgICB0cnVlX2xhYmVsX2ZyYW1lID0gdG9yY2guY2F0KFt0b3JjaC5jYXQoKHRydWVfbGFiZWxfZnJhbWUsIHRydWVfbGFiZWxfc2VnbWVudFtpXS5yZXBlYXQobGVuZ3Roc192YXJ5aW5nX3NlZ21lbnRbaV0sMSkpLCBkaW09MCkgZm9yIGkgaW4gcmFuZ2Uobl9ldmVudCldKQ0KICAgIGFzc2VydCB0cnVlX2xhYmVsX2ZyYW1lLnNoYXBlWzBdID09IG5fZnJhbWUNCg0KICAgIGZyYW1lX2FjYyA9ICgocHJlZGljdGVkX2xhYmVsX2ZyYW1lID09IHRydWVfbGFiZWxfZnJhbWUpLnN1bSgpKS5kb3VibGUoKSAqIDEwMCAvIG5fZnJhbWUuZG91YmxlKCkNCiAgICBmcmFtZV9taUYgPSBmMV9zY29yZSh0cnVlX2xhYmVsX2ZyYW1lLCBwcmVkaWN0ZWRfbGFiZWxfZnJhbWUsIGF2ZXJhZ2U9J21pY3JvJykgKiAxMDANCiAgICBmcmFtZV9tYUYgPSBmMV9zY29yZSh0cnVlX2xhYmVsX2ZyYW1lLCBwcmVkaWN0ZWRfbGFiZWxfZnJhbWUsIGF2ZXJhZ2U9J3dlaWdodGVkJykgKiAxMDANCg0KICAgIGV2ZW50X2FjYyA9IGV2ZW50X2FjYy5pdGVtKCkNCiAgICBmcmFtZV9hY2MgPSBmcmFtZV9hY2MuaXRlbSgpDQogICAgcmV0dXJuIGV2ZW50X2FjYywgZXZlbnRfbWlGLCBldmVudF9tYUYsIGZyYW1lX2FjYywgZnJhbWVfbWlGLCBmcmFtZV9tYUYNCg0K"}, "code/stage2_ref/lambda.json": {"sha256": "956983c6e65b4cc46582f0d816ac0ed444b30ed083d93c8658bbebf5fbb4290f", "b64": "ew0KICAibGFtIjogNjAxMi4zODExNzcwMjc2MzI1LA0KICAicnVsZSI6ICJsYW0gPSAwLjEgKiBtZWFuIG1haW4gbG9zcyBwZXIgYmF0Y2ggLyBtZWFuIExfWFBDIHBlciBiYXRjaCBhdCBpbml0aWFsaXNhdGlvbiIsDQogICJtZWFzdXJlZF9vbiI6ICJzb3VyY2UgcGVyc29ucyBvZiB0YXJnZXQgMCwgc2VlZCAwLCBvbmUgZXBvY2ggb2YgYmFsYW5jZWQgYmF0Y2hlcywgbm8gcGFyYW1ldGVyIHVwZGF0ZXMsIG5vIHRhcmdldCBkYXRhIiwNCiAgIm1lYW5fbWFpbl9sb3NzX3Blcl9iYXRjaCI6IDI0NjIwMi4zNTIxNjM0NjE1MywNCiAgIm1lYW5feHBjX3Blcl9iYXRjaCI6IDQuMDk0OTIyNTQyNTcyMDIxNSwNCiAgInRhdSI6IDAuMSwNCiAgIm5fYmF0Y2hlcyI6IDEzLA0KICAibWVhbl9hbmNob3JzX3Blcl9iYXRjaCI6IDY0LjAsDQogICJtZWFuX3Bvc2l0aXZlc19wZXJfYW5jaG9yIjogNy45NzExNTM4NDYxNTM4NDYsDQogICJmYWtlX2RhdGEiOiBmYWxzZSwNCiAgInRvcmNoIjogIjIuMi4yK2NwdSIsDQogICJkZXZpY2UiOiAiY3B1Ig0KfQ=="}, "code/stage2_ref/run_stage2.py": {"sha256": "0566125bd8358a419849de2528553bfdf96a962b538403a25f9fbc938ae7c07d", "b64": "..."}, "code/stage2_ref/xpc.py": {"sha256": "df28ef150b7abc0116b59cf0740e0225dd823e0851e27180798957dfa4a83b62", "b64": "IiIiU3RhZ2UgMiAoWFBDLUIpIGJ1aWxkaW5nIGJsb2NrczogcGVyc29uLWJhbGFuY2VkIGJhdGNoZXMgYW5kIHRoZSBjcm9zcy1wZXJzb24gc3VwZXJ2aXNlZCBjb250cmFzdGl2ZSBsb3NzLgpLZXB0IGluIGl0cyBvd24gbW9kdWxlIHNvIGNoZWNrcy90ZXN0X3N0YWdlMi5weSBjYW4gdGVzdCB0aGVtIGRpcmVjdGx5LiBOb3RoaW5nIGhlcmUgdG91Y2hlcyBzdGFnZS0xIGNvZGUuIiIiCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyLCBXZWlnaHRlZFJhbmRvbVNhbXBsZXIKCgpjbGFzcyBCYWxhbmNlZEJhdGNoZXM6CiAgICAiIiJQZXJzb24tYmFsYW5jZWQgc291cmNlIGJhdGNoZXM6IGV2ZXJ5IHN0ZXAgY29uY2F0ZW5hdGVzIGJhdGNoX3NpemUvSyB3aW5kb3dzIGZyb20gZWFjaCBvZiB0aGUgSyBzb3VyY2UKICAgIHBlcnNvbnMuIFBlciBwZXJzb24gaXQgdXNlcyB0aGUgc2FtZSBjbGFzcy1iYWxhbmNlZCBXZWlnaHRlZFJhbmRvbVNhbXBsZXIgYXMgc3RhZ2UgMSAod2l0aCByZXBsYWNlbWVudCwKICAgIHdlaWdodHMgMTAwL2NsYXNzIGNvdW50LCBnZW5lcmF0b3Igc2VlZCA9IHNlZWQqMTAwICsgayksIGRyYXduIGJhdGNoX3NpemUvSyBhdCBhIHRpbWU7IGEgcGVyc29uJ3Mgc3RyZWFtCiAgICByZXN0YXJ0cyB3aGVuIGV4aGF1c3RlZC4gU3RlcHMgcGVyIGVwb2NoID0gc3VtX3AgZmxvb3Iobl9wIC8gYmF0Y2hfc2l6ZSkgPSB0aGUgc3RhZ2UtMSBudW1iZXIsIHNvIGV2ZXJ5CiAgICB2YXJpYW50IHNlZXMgdGhlIHNhbWUgbnVtYmVyIG9mIGdyYWRpZW50IHN0ZXBzIGFuZCB3aW5kb3dzIHBlciBlcG9jaC4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwYXJ0cywgc2VlZCwgYmF0Y2hfc2l6ZSwgbl9jbGFzcywgbWFrZV9kYXRhc2V0KToKICAgICAgICBLID0gbGVuKHBhcnRzKQogICAgICAgIGFzc2VydCBiYXRjaF9zaXplICUgSyA9PSAwLCAnYmF0Y2ggc2l6ZSBtdXN0IGRpdmlkZSBldmVubHkgYWNyb3NzIHNvdXJjZSBwZXJzb25zJwogICAgICAgIHNlbGYucGVyID0gYmF0Y2hfc2l6ZSAvLyBLCiAgICAgICAgc2VsZi5sb2FkZXJzLCBzZWxmLmdlbnMgPSBbXSwgW10KICAgICAgICBmb3IgaywgKFgsIHksIGQpIGluIGVudW1lcmF0ZShwYXJ0cyk6CiAgICAgICAgICAgIGNvdW50cyA9IG5wLmJpbmNvdW50KHksIG1pbmxlbmd0aD1uX2NsYXNzKS5hc3R5cGUoZmxvYXQpCiAgICAgICAgICAgIHcgPSBucC53aGVyZShjb3VudHNbeV0gPiAwLCAxMDAuMCAvIGNvdW50c1t5XSwgMC4wKQogICAgICAgICAgICBnID0gdG9yY2guR2VuZXJhdG9yKCk7IGcubWFudWFsX3NlZWQoc2VlZCAqIDEwMCArIGspCiAgICAgICAgICAgIHNhbXBsZXIgPSBXZWlnaHRlZFJhbmRvbVNhbXBsZXIodG9yY2guYXNfdGVuc29yKHcsIGR0eXBlPXRvcmNoLmRvdWJsZSksIGxlbih3KSwgcmVwbGFjZW1lbnQ9VHJ1ZSwgZ2VuZXJhdG9yPWcpCiAgICAgICAgICAgIHNlbGYubG9hZGVycy5hcHBlbmQoRGF0YUxvYWRlcihtYWtlX2RhdGFzZXQoWCwgeSwgZCksIGJhdGNoX3NpemU9c2VsZi5wZXIsIHNhbXBsZXI9c2FtcGxlciwgZHJvcF9sYXN0PVRydWUpKQogICAgICAgICAgICBzZWxmLmdlbnMuYXBwZW5kKGcpCiAgICAgICAgc2VsZi5zdGVwcyA9IHN1bShsZW4oeSkgLy8gYmF0Y2hfc2l6ZSBmb3IgXywgeSwgXyBpbiBwYXJ0cykKICAgICAgICAjIFJFU1VNRSBTVVBQT1JUICh0cmFuc2ZlciBwYWNrYWdlKTogZm9yIGVhY2ggcGVyc29uIHN0cmVhbSByZW1lbWJlciB0aGUgc2FtcGxlci1nZW5lcmF0b3Igc3RhdGUgYXQgdGhlIG1vbWVudAogICAgICAgICMgaXRzIGN1cnJlbnQgaXRlcmF0b3Igd2FzIGNyZWF0ZWQsIGFuZCBob3cgbWFueSBiYXRjaGVzIGhhdmUgYmVlbiBkcmF3biBmcm9tIGl0LiBSZWFkaW5nIGEgZ2VuZXJhdG9yIHN0YXRlCiAgICAgICAgIyBkb2VzIG5vdCBjaGFuZ2UgaXQsIHNvIHVuLXJlc3VtZWQgYmVoYXZpb3VyIGlzIGlkZW50aWNhbCB0byB0aGUgb3JpZ2luYWwgbW9kdWxlLgogICAgICAgIHNlbGYuaXRlcnMsIHNlbGYuc25hcCwgc2VsZi5jb25zdW1lZCA9IFtOb25lXSAqIEssIFtOb25lXSAqIEssIFswXSAqIEsKICAgICAgICBmb3IgayBpbiByYW5nZShLKToKICAgICAgICAgICAgc2VsZi5fbmV3X2l0ZXIoaykKCiAgICBkZWYgX25ld19pdGVyKHNlbGYsIGspOgogICAgICAgIHNlbGYuc25hcFtrXSA9IHNlbGYuZ2Vuc1trXS5nZXRfc3RhdGUoKS5jbG9uZSgpCiAgICAgICAgc2VsZi5pdGVyc1trXSA9IGl0ZXIoc2VsZi5sb2FkZXJzW2tdKQogICAgICAgIHNlbGYuY29uc3VtZWRba10gPSAwCgogICAgZGVmIF9uZXh0KHNlbGYsIGspOgogICAgICAgIHRyeToKICAgICAgICAgICAgYiA9IG5leHQoc2VsZi5pdGVyc1trXSkKICAgICAgICBleGNlcHQgU3RvcEl0ZXJhdGlvbjoKICAgICAgICAgICAgc2VsZi5fbmV3X2l0ZXIoaykKICAgICAgICAgICAgYiA9IG5leHQoc2VsZi5pdGVyc1trXSkKICAgICAgICBzZWxmLmNvbnN1bWVkW2tdICs9IDEKICAgICAgICByZXR1cm4gYgoKICAgIGRlZiBzdGF0ZV9kaWN0KHNlbGYpOgogICAgICAgIHJldHVybiB7J3NuYXAnOiBbcy5jbG9uZSgpIGZvciBzIGluIHNlbGYuc25hcF0sICdjb25zdW1lZCc6IGxpc3Qoc2VsZi5jb25zdW1lZCl9CgogICAgZGVmIGxvYWRfc3RhdGVfZGljdChzZWxmLCBzdCk6CiAgICAgICAgIiIiUmVwbGF5IGVhY2ggcGVyc29uIHN0cmVhbSB0byB0aGUgc2F2ZWQgcG9zaXRpb24uIFRoZSByZXBsYXkgcnVucyB1bmRlciBhIGZvcmtlZCBnbG9iYWwgUk5HLCBiZWNhdXNlCiAgICAgICAgY3JlYXRpbmcgYSBEYXRhTG9hZGVyIGl0ZXJhdG9yIGRyYXdzIGEgYmFzZSBzZWVkIGZyb20gdGhlIGdsb2JhbCB0b3JjaCBSTkcsIGFuZCB0aGF0IGRyYXcgYWxyZWFkeSBoYXBwZW5lZAogICAgICAgIChhbmQgaXMgcmVmbGVjdGVkIGluIHRoZSByZXN0b3JlZCBnbG9iYWwgUk5HIHN0YXRlKSBpbiB0aGUgb3JpZ2luYWwgcnVuLiIiIgogICAgICAgIHdpdGggdG9yY2gucmFuZG9tLmZvcmtfcm5nKGRldmljZXM9W10pOgogICAgICAgICAgICBmb3IgayBpbiByYW5nZShsZW4oc2VsZi5sb2FkZXJzKSk6CiAgICAgICAgICAgICAgICBzZWxmLmdlbnNba10uc2V0X3N0YXRlKHN0WydzbmFwJ11ba10pCiAgICAgICAgICAgICAgICBzZWxmLnNuYXBba10gPSBzdFsnc25hcCddW2tdLmNsb25lKCkKICAgICAgICAgICAgICAgIHNlbGYuaXRlcnNba10gPSBpdGVyKHNlbGYubG9hZGVyc1trXSkKICAgICAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKHN0Wydjb25zdW1lZCddW2tdKToKICAgICAgICAgICAgICAgICAgICBuZXh0KHNlbGYuaXRlcnNba10pCiAgICAgICAgICAgICAgICBzZWxmLmNvbnN1bWVkW2tdID0gc3RbJ2NvbnN1bWVkJ11ba10KCiAgICBkZWYgX19pdGVyX18oc2VsZik6CiAgICAgICAgZm9yIF8gaW4gcmFuZ2Uoc2VsZi5zdGVwcyk6CiAgICAgICAgICAgIGJzID0gW3NlbGYuX25leHQoaykgZm9yIGsgaW4gcmFuZ2UobGVuKHNlbGYubG9hZGVycykpXQogICAgICAgICAgICB5aWVsZCB0dXBsZSh0b3JjaC5jYXQoW2JbaV0gZm9yIGIgaW4gYnNdKSBmb3IgaSBpbiByYW5nZSgzKSkKCgpkZWYgbWFrZV9wcm9qX2hlYWQoel9kaW0sIHNlZWQpOgogICAgIiIiVHJhaW5pbmctb25seSBwcm9qZWN0aW9uIGhlYWQgZzogeiAtPiA2NCAtPiAzMiwgYnVpbHQgb24gdGhlIENQVS4gVGhlIENQVSBhbmQgQUxMIENVREEgcmFuZG9tIHN0YXRlcyBhcmUgc2F2ZWQKICAgIGFuZCByZXN0b3JlZCBhcm91bmQgaXRzIGNvbnN0cnVjdGlvbiwgc28gY3JlYXRpbmcgaXQgZG9lcyBub3Qgc2hpZnQgYW55IGdsb2JhbCByYW5kb20gc3RyZWFtOiB0aGUgbW9kZWwncwogICAgc2FtcGxpbmcgbm9pc2UgaXMgaWRlbnRpY2FsIHRvIHN0YWdlIDEsIGFuZCBpZV9vZmZfYmFsIC8geHBjX2JhbCBzaGFyZSBpZGVudGljYWwgaW5pdCBhbmQgbm9pc2UuCiAgICAoQ29ycmVjdGlvbiBmb3VuZCBieSB0aGUgQ29sYWIgdGVzdCBydW46IHRvcmNoLm1hbnVhbF9zZWVkIGFsc28gcmVzZWVkcyBDVURBLCBzbyBmb3JraW5nIG9ubHkgdGhlIENQVSBSTkcKICAgIHdhcyBub3QgZW5vdWdoIG9uIGEgR1BVLikiIiIKICAgIGN1ZGFfZGV2cyA9IGxpc3QocmFuZ2UodG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSkpIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSBbXQogICAgd2l0aCB0b3JjaC5yYW5kb20uZm9ya19ybmcoZGV2aWNlcz1jdWRhX2RldnMpOgogICAgICAgIHRvcmNoLm1hbnVhbF9zZWVkKDEwXzAwMCArIHNlZWQpCiAgICAgICAgcmV0dXJuIG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKHpfZGltLCA2NCksIG5uLlJlTFUoKSwgbm4uTGluZWFyKDY0LCAzMikpCgoKZGVmIHhwY19tYXNrcyh5LCBkKToKICAgIHNhbWVfeSA9IHlbOiwgTm9uZV0gPT0geVtOb25lLCA6XQogICAgc2FtZV9kID0gZFs6LCBOb25lXSA9PSBkW05vbmUsIDpdCiAgICBwb3MgPSBzYW1lX3kgJiB+c2FtZV9kICAgICAgICAgICAgICAgICAjIHNhbWUgYWN0aXZpdHksIGRpZmZlcmVudCBwZXJzb24KICAgIGRlbm9tX29rID0gfihzYW1lX3kgJiBzYW1lX2QpICAgICAgICAgICMgZHJvcHMgc2VsZiBhbmQgc2FtZS1wZXJzb24tc2FtZS1hY3Rpdml0eSAob3ZlcmxhcHBpbmcgbmVhci1kdXBsaWNhdGVzKQogICAgcmV0dXJuIHBvcywgZGVub21fb2sKCgpkZWYgeHBjX2xvc3MobXUsIHksIGQsIGhlYWQsIHRhdSk6CiAgICAiIiJDcm9zcy1wZXJzb24gc3VwZXJ2aXNlZCBjb250cmFzdGl2ZSBsb3NzIChTdXBDb24gZm9ybSwgS2hvc2xhIGV0IGFsLiAyMDIwKSBvbiB0aGUgYWN0aXZpdHktY29kZSBtZWFuIG11LgogICAgTCA9IG1lYW4gb3ZlciBhbmNob3JzIGkgd2l0aCB8UChpKXw+PTEgb2YgIC0xL3xQKGkpfCBzdW1fe3AgaW4gUChpKX0gbG9nKCBleHAodV9pLnVfcC90YXUpIC8gc3VtX3thIGluIEEoaSl9IGV4cCh1X2kudV9hL3RhdSkgKQogICAgUmV0dXJucyAobG9zcywgbl9hbmNob3JzLCBtZWFuIHBvc2l0aXZlcyBwZXIgYW5jaG9yKS4iIiIKICAgIHUgPSBGLm5vcm1hbGl6ZShoZWFkKG11KSwgZGltPTEpCiAgICBzaW0gPSB1IEAgdS50KCkgLyB0YXUKICAgIHBvcywgZGVub21fb2sgPSB4cGNfbWFza3MoeSwgZCkKICAgIHBvcyA9IHBvcy5mbG9hdCgpCiAgICBsb2dfZGVuID0gdG9yY2gubG9nc3VtZXhwKHNpbS5tYXNrZWRfZmlsbCh+ZGVub21fb2ssIGZsb2F0KCctaW5mJykpLCBkaW09MSwga2VlcGRpbT1UcnVlKQogICAgbG9nX3Byb2IgPSBzaW0gLSBsb2dfZGVuCiAgICBuX3BvcyA9IHBvcy5zdW0oMSkKICAgIGhhcyA9IG5fcG9zID4gMAogICAgaWYgbm90IGJvb2woaGFzLmFueSgpKToKICAgICAgICByZXR1cm4gbXUuc3VtKCkgKiAwLjAsIDAsIDAuMAogICAgcGVyX2FuY2hvciA9IC0obG9nX3Byb2IgKiBwb3MpLnN1bSgxKVtoYXNdIC8gbl9wb3NbaGFzXQogICAgcmV0dXJuIHBlcl9hbmNob3IubWVhbigpLCBpbnQoaGFzLnN1bSgpKSwgZmxvYXQobl9wb3NbaGFzXS5tZWFuKCkpCg=="}, "code/stage3/p3_variants.py": {"sha256": "1d52bfffe11fa1a2b6316c8a50327a83a9e2ae648b8329daa8ccc4cad5b6b402", "b64": "IiIiUGVyc29uIDMsIHN0YWdlIDM6IHRoZSB0d28gaXNvbGF0ZWQgY2hhbmdlcyB0ZXN0ZWQgYWdhaW5zdCB0aGUgc3RhZ2UtMSBjb250cm9sIChzZWUgLi4vLi4vUFJFREVDTEFSRURfUExBTi5tZCkuCgpDICBhcHBseV9sYXlvdXRfZml4KG1vZGVsKTogcmVwbGFjZXMgZm9yd2FyZCgpIG9mIHRoZSBlbmNvZGVyIElOU1RBTkNFUyBxemQgLyBxenkgKGFuZCBxenggaWYgcHJlc2VudCkgd2l0aCBhbiBleGFjdAogICBjb3B5IG9mIHRoZSByZWxlYXNlZCBmb3J3YXJkIGluIHdoaWNoIHRoZSBpbnB1dCByZXNoYXBpbmcKICAgICAgICB4LnZpZXcoLTEsIHguc2hhcGVbMl0sIDEsIHguc2hhcGVbMV0pICAgICAgIyByZWxlYXNlZDogcmVpbnRlcnByZXRzIG1lbW9yeSAtPiBzZW5zb3JzIGFuZCB0aW1lIG1peGVkCiAgIGlzIHJlcGxhY2VkIGJ5CiAgICAgICAgeC5wZXJtdXRlKDAsIDIsIDMsIDEpICAgICAgICAgICAgICAgICAgICAgICAjIChCLDEyOCw5LDEpIC0+IChCLDksMSwxMjgpOiBjaGFubmVsID0gc2Vuc29yLCB3aWR0aCA9IHRpbWUKICAgTm90aGluZyBlbHNlIGNoYW5nZXMgKHNhbWUgcGFyYW1ldGVycywgc2FtZSBpbml0LCBzYW1lIGRlY29kZXIpLiBtb2RlbF9HSUxFX3VjaWhhci5weSBpdHNlbGYgaXMgTk9UIGVkaXRlZC4KCkIgIFBlcnNvblN0eWxlQXVnOiB0cmFpbmluZy10aW1lIGF1Z21lbnRhdGlvbiBpbiByYXcgcGh5c2ljYWwgdW5pdHMgKFVDSSBIQVIgY2hhbm5lbCBvcmRlcgogICBbYm9keV9hY2MgeHl6LCBib2R5X2d5cm8geHl6LCB0b3RhbF9hY2MgeHl6XSksIGVhY2ggd2luZG93IHRyYW5zZm9ybWVkIHdpdGggcHJvYmFiaWxpdHkgcCA9IDAuNToKICAgICAtIG1vdmVtZW50LWludGVuc2l0eSBzY2FsaW5nIHMgfiBOKDEsIDAuMV4yKSBjbGlwcGVkIHRvIFswLjcsIDEuM10gb24gYm9keV9hY2MgYW5kIGJvZHlfZ3lybzsKICAgICAgIHRvdGFsX2FjYyArPSAocyAtIDEpICogYm9keV9hY2MgKGdyYXZpdHkgY29tcG9uZW50IHVuY2hhbmdlZCkKICAgICAtIG9uZSBzbWFsbCByb3RhdGlvbiAoYXhpcyB1bmlmb3JtIG9uIHRoZSBzcGhlcmUsIGFuZ2xlIH4gVSgtMTUsICsxNSkgZGVnKSBhcHBsaWVkIHRvIGFsbCB0aHJlZSB0cmlhZHMKICAgICAtIEdhdXNzaWFuIG5vaXNlIHNpZ21hID0gMC4wMSBvbiBib2R5X2FjYyAoc2FtZSBub2lzZSBhZGRlZCB0byB0b3RhbF9hY2MpIGFuZCBvbiBib2R5X2d5cm8KICAgSXQgb3ducyBhIHNlcGFyYXRlIHRvcmNoLkdlbmVyYXRvciBhbmQgZHJhd3MgYSBGSVhFRCBudW1iZXIgb2YgcmFuZG9tIG51bWJlcnMgcGVyIHdpbmRvdyAod2hldGhlciBvciBub3QgdGhlIHdpbmRvdyBpcwogICB0cmFuc2Zvcm1lZCksIHNvIHRoZSBtb2RlbCBpbml0aWFsaXNhdGlvbiwgYmF0Y2ggb3JkZXIgYW5kIFZBRSBzYW1wbGluZyBub2lzZSBhcmUgaWRlbnRpY2FsIHRvIHRoZSBjb250cm9sLgoiIiIKaW1wb3J0IG1hdGgKaW1wb3J0IHR5cGVzCgppbXBvcnQgdG9yY2gKCkJBLCBCRywgVEEgPSBzbGljZSgwLCAzKSwgc2xpY2UoMywgNiksIHNsaWNlKDYsIDkpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBDOiBsYXlvdXQgZml4CmRlZiB0b19pbWFnZV9yZWxlYXNlZCh4KToKICAgIHJldHVybiB4LnZpZXcoLTEsIHguc2hhcGVbMl0sIDEsIHguc2hhcGVbMV0pCgoKZGVmIHRvX2ltYWdlX2ZpeGVkKHgpOgogICAgcmV0dXJuIHgucGVybXV0ZSgwLCAyLCAzLCAxKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyAoQiwgVCwgQywgMSkgLT4gKEIsIEMsIDEsIFQpCgoKZGVmIF9maXhlZF9mb3J3YXJkKHNlbGYsIHgpOgogICAgIyBleGFjdCBjb3B5IG9mIHF6ZC9xengvcXp5LmZvcndhcmQgaW4gbW9kZWxfR0lMRV91Y2loYXIucHkgZXhjZXB0IHRoZSBtYXJrZWQgbGluZQogICAgeCA9IHguZmxvYXQoKQogICAgeF9pbWcgPSB0b19pbWFnZV9maXhlZCh4KSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgPC0tIHRoZSBvbmx5IGNoYW5nZSAod2FzIC52aWV3KC4uLikpCgogICAgb3V0X2NvbnYxID0gc2VsZi5jb252MSh4X2ltZykKICAgIG91dDEsIGlkeDEgPSBzZWxmLnBvb2wxKG91dF9jb252MSkKCiAgICBvdXRfY29udjIgPSBzZWxmLmNvbnYyKG91dDEpCiAgICBvdXQyLCBpZHgyID0gc2VsZi5wb29sMihvdXRfY29udjIpCgogICAgb3V0X2NvbnYzID0gc2VsZi5jb252MyhvdXQyKQogICAgb3V0MywgaWR4MyA9IHNlbGYucG9vbDMob3V0X2NvbnYzKQoKICAgIG91dF9jb252NCA9IHNlbGYuY29udjQob3V0MykKICAgIG91dDQsIGlkeDQgPSBzZWxmLnBvb2w0KG91dF9jb252NCkKCiAgICBvdXQgPSBvdXQ0LnJlc2hhcGUoLTEsIG91dDQuc2hhcGVbMV0gKiBvdXQ0LnNoYXBlWzNdKQogICAgc2l6ZTEgPSBvdXQxLnNpemUoKQogICAgc2l6ZTIgPSBvdXQyLnNpemUoKQogICAgc2l6ZTMgPSBvdXQzLnNpemUoKQogICAgc2l6ZTQgPSBvdXQ0LnNpemUoKQoKICAgIGxvYyA9IHNlbGYuZmMxMShvdXQpCiAgICBzY2FsZSA9IHNlbGYuZmMxMihvdXQpICsgMWUtNwoKICAgIHJldHVybiBsb2MsIHNjYWxlLCBbaWR4MSwgaWR4MiwgaWR4MywgaWR4NF0sIFtzaXplMSwgc2l6ZTIsIHNpemUzLCBzaXplNF0KCgpkZWYgYXBwbHlfbGF5b3V0X2ZpeChtb2RlbCk6CiAgICBwYXRjaGVkID0gW10KICAgIGZvciBuYW1lIGluICgncXpkJywgJ3F6eCcsICdxenknKToKICAgICAgICBlbmMgPSBnZXRhdHRyKG1vZGVsLCBuYW1lLCBOb25lKQogICAgICAgIGlmIGVuYyBpcyBub3QgTm9uZToKICAgICAgICAgICAgZW5jLmZvcndhcmQgPSB0eXBlcy5NZXRob2RUeXBlKF9maXhlZF9mb3J3YXJkLCBlbmMpCiAgICAgICAgICAgIHBhdGNoZWQuYXBwZW5kKG5hbWUpCiAgICByZXR1cm4gcGF0Y2hlZAoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gQjogYXVnbWVudGF0aW9uCmRlZiBfcm90YXRpb25zKGF4aXMsIGFuZ2xlKToKICAgICIiIlJvZHJpZ3VlczogYXhpcyAoTiwzKSB1bml0IHZlY3RvcnMsIGFuZ2xlIChOLCkgcmFkaWFucyAtPiAoTiwzLDMpLiIiIgogICAga3gsIGt5LCBreiA9IGF4aXNbOiwgMF0sIGF4aXNbOiwgMV0sIGF4aXNbOiwgMl0KICAgIHogPSB0b3JjaC56ZXJvc19saWtlKGt4KQogICAgSyA9IHRvcmNoLnN0YWNrKFt0b3JjaC5zdGFjayhbeiwgLWt6LCBreV0sIDEpLCB0b3JjaC5zdGFjayhba3osIHosIC1reF0sIDEpLCB0b3JjaC5zdGFjayhbLWt5LCBreCwgel0sIDEpXSwgMSkKICAgIHMsIGMgPSB0b3JjaC5zaW4oYW5nbGUpWzosIE5vbmUsIE5vbmVdLCB0b3JjaC5jb3MoYW5nbGUpWzosIE5vbmUsIE5vbmVdCiAgICBleWUgPSB0b3JjaC5leWUoMywgZHR5cGU9YXhpcy5kdHlwZSkuZXhwYW5kKGxlbihhbmdsZSksIDMsIDMpCiAgICByZXR1cm4gZXllICsgcyAqIEsgKyAoMSAtIGMpICogKEsgQCBLKQoKCmNsYXNzIFBlcnNvblN0eWxlQXVnOgogICAgUCwgU0NBTEVfU0QsIFNDQUxFX0NMSVAsIE1BWF9ERUcsIE5PSVNFX1NEID0gMC41LCAwLjEsICgwLjcsIDEuMyksIDE1LjAsIDAuMDEgICAjIGZpeGVkIGEgcHJpb3JpIChwbGFuIHNlY3Rpb24gMSkKCiAgICBkZWYgX19pbml0X18oc2VsZiwgc2VlZCk6CiAgICAgICAgc2VsZi5nID0gdG9yY2guR2VuZXJhdG9yKCkKICAgICAgICBzZWxmLmcubWFudWFsX3NlZWQoc2VlZCAqIDc5MTkgKyAxNykKCiAgICBkZWYgc3RhdGVfZGljdChzZWxmKToKICAgICAgICByZXR1cm4geydnJzogc2VsZi5nLmdldF9zdGF0ZSgpLmNsb25lKCl9CgogICAgZGVmIGxvYWRfc3RhdGVfZGljdChzZWxmLCBzdCk6CiAgICAgICAgc2VsZi5nLnNldF9zdGF0ZShzdFsnZyddKQoKICAgIGRlZiBkcmF3cyhzZWxmLCBuLCBUKToKICAgICAgICBnID0gc2VsZi5nCiAgICAgICAgYXBwbHkgPSB0b3JjaC5yYW5kKG4sIGdlbmVyYXRvcj1nKSA8IHNlbGYuUAogICAgICAgIHMgPSAoMSArIHNlbGYuU0NBTEVfU0QgKiB0b3JjaC5yYW5kbihuLCBnZW5lcmF0b3I9ZywgZHR5cGU9dG9yY2guZmxvYXQ2NCkpLmNsYW1wKCpzZWxmLlNDQUxFX0NMSVApCiAgICAgICAgYXhpcyA9IHRvcmNoLnJhbmRuKG4sIDMsIGdlbmVyYXRvcj1nLCBkdHlwZT10b3JjaC5mbG9hdDY0KQogICAgICAgIGF4aXMgPSBheGlzIC8gYXhpcy5ub3JtKGRpbT0xLCBrZWVwZGltPVRydWUpLmNsYW1wX21pbigxZS0xMikKICAgICAgICBhbmcgPSAodG9yY2gucmFuZChuLCBnZW5lcmF0b3I9ZywgZHR5cGU9dG9yY2guZmxvYXQ2NCkgKiAyIC0gMSkgKiBtYXRoLnJhZGlhbnMoc2VsZi5NQVhfREVHKQogICAgICAgIG5fYWNjID0gc2VsZi5OT0lTRV9TRCAqIHRvcmNoLnJhbmRuKG4sIFQsIDMsIGdlbmVyYXRvcj1nLCBkdHlwZT10b3JjaC5mbG9hdDY0KQogICAgICAgIG5fZ3lyID0gc2VsZi5OT0lTRV9TRCAqIHRvcmNoLnJhbmRuKG4sIFQsIDMsIGdlbmVyYXRvcj1nLCBkdHlwZT10b3JjaC5mbG9hdDY0KQogICAgICAgIHJldHVybiBhcHBseSwgcywgYXhpcywgYW5nLCBuX2FjYywgbl9neXIKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgdHJhbnNmb3JtKHgsIGFwcGx5LCBzLCBheGlzLCBhbmcsIG5fYWNjLCBuX2d5cik6CiAgICAgICAgIiIieDogKE4sIFQsIDlbLCAxXSkgZmxvYXQgdGVuc29yIChDUFUpLiBSZXR1cm5zIGEgbmV3IHRlbnNvcjsgd2luZG93cyB3aXRoIGFwcGx5PUZhbHNlIGFyZSByZXR1cm5lZCB1bmNoYW5nZWQuIiIiCiAgICAgICAgc3F1ZWV6ZSA9IHguZGltKCkgPT0gNAogICAgICAgIFggPSAoeFsuLi4sIDBdIGlmIHNxdWVlemUgZWxzZSB4KS50byh0b3JjaC5mbG9hdDY0KQogICAgICAgIGJhLCBiZywgdGEgPSBYWy4uLiwgQkFdLCBYWy4uLiwgQkddLCBYWy4uLiwgVEFdCiAgICAgICAgZ3JhdiA9IHRhIC0gYmEgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBncmF2aXR5ICgrIHJlc2lkdWFsKSBjb21wb25lbnQsIGtlcHQgYXMgaXMKICAgICAgICBzdiA9IHNbOiwgTm9uZSwgTm9uZV0KICAgICAgICBiYTIgPSBzdiAqIGJhICsgbl9hY2MKICAgICAgICBiZzIgPSBzdiAqIGJnICsgbl9neXIKICAgICAgICB0YTIgPSBncmF2ICsgYmEyICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjID0gdGEgKyAocy0xKSpiYSArIG5vaXNlCiAgICAgICAgUiA9IF9yb3RhdGlvbnMoYXhpcywgYW5nKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyAoTiwzLDMpOyByb3RhdGUgcm93IHZlY3RvcnM6IHYgQCBSXlQKICAgICAgICByb3QgPSBsYW1iZGEgdjogdiBAIFIudHJhbnNwb3NlKDEsIDIpCiAgICAgICAgb3V0ID0gdG9yY2guY2F0KFtyb3QoYmEyKSwgcm90KGJnMiksIHJvdCh0YTIpXSwgZGltPS0xKQogICAgICAgIG91dCA9IHRvcmNoLndoZXJlKGFwcGx5WzosIE5vbmUsIE5vbmVdLCBvdXQsIFgpCiAgICAgICAgb3V0ID0gb3V0LnRvKHguZHR5cGUpCiAgICAgICAgcmV0dXJuIG91dFsuLi4sIE5vbmVdIGlmIHNxdWVlemUgZWxzZSBvdXQKCiAgICBkZWYgX19jYWxsX18oc2VsZiwgeCk6CiAgICAgICAgZCA9IHNlbGYuZHJhd3MoeC5zaGFwZVswXSwgeC5zaGFwZVsxXSkKICAgICAgICByZXR1cm4gc2VsZi50cmFuc2Zvcm0oeC5jcHUoKSwgKmQpLnRvKHguZGV2aWNlKQo="}, "code/stage3/run_stage3.py": {"sha256": "0f4414caa2fb7bdb37e496d6c5da2ce0ccd31296f4cb79cdede5833fae5d85f6", "b64": "..."}, "code/stage3/checks/test_stage3.py": {"sha256": "18383c7a74ba20e48949b2af0335c93588fc4e4381c879f7577024c39dfa124d", "b64": "IiIiU3RhZ2UtMyBjb3JyZWN0bmVzcyBjaGVja3MgKENQVSBvciBHUFUpLiBQcmludHMgUEFTUy9GQUlMIHBlciBjaGVjayBhbmQgJ0FMTCBQQVNTRUQnIGF0IHRoZSBlbmQuClVzYWdlIChmcm9tIGFueXdoZXJlKTogcHl0aG9uIGNvZGUvc3RhZ2UzL2NoZWNrcy90ZXN0X3N0YWdlMy5weSIiIgppbXBvcnQganNvbiwgbWF0aCwgb3MsIHNodXRpbCwgc3VicHJvY2Vzcywgc3lzCgpIRVJFID0gb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpOyBTMyA9IG9zLnBhdGguZGlybmFtZShIRVJFKTsgQ09ERSA9IG9zLnBhdGguZGlybmFtZShTMykKc3lzLnBhdGguaW5zZXJ0KDAsIENPREUpOyBzeXMucGF0aC5pbnNlcnQoMCwgUzMpCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgppZiBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgIG5uLk1vZHVsZS5jdWRhID0gbGFtYmRhIHNlbGYsICphLCAqKms6IHNlbGYKZnJvbSBwM192YXJpYW50cyBpbXBvcnQgYXBwbHlfbGF5b3V0X2ZpeCwgUGVyc29uU3R5bGVBdWcsIHRvX2ltYWdlX3JlbGVhc2VkLCB0b19pbWFnZV9maXhlZCwgX3JvdGF0aW9ucwppbXBvcnQgYXJncGFyc2UKZnJvbSBtb2RlbF9HSUxFX3VjaWhhciBpbXBvcnQgR0lMRQoKcmVzdWx0cyA9IFtdCmRlZiBjaGVjayhuYW1lLCBvaywgaW5mbz0nJyk6CiAgICByZXN1bHRzLmFwcGVuZChib29sKG9rKSk7IHByaW50KCgnUEFTUyAnIGlmIG9rIGVsc2UgJ0ZBSUwgJykgKyBuYW1lICsgKGYnICBbe2luZm99XScgaWYgaW5mbyBlbHNlICcnKSwgZmx1c2g9VHJ1ZSkKCkRFViA9IHRvcmNoLmRldmljZSgnY3VkYTowJyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgJ2NwdScpCkIsIFQsIEMgPSA0LCAxMjgsIDkKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBDMTogd2hhdCB0aGUgcmVsZWFzZWQgLnZpZXcoKSBkb2VzCnggPSB0b3JjaC5hcmFuZ2UoQiAqIFQgKiBDLCBkdHlwZT10b3JjaC5mbG9hdDMyKS52aWV3KEIsIFQsIEMsIDEpICAgICMgdmFsdWUgZW5jb2RlcyAoYiwgdCwgYykKZGVjID0gbGFtYmRhIHY6ICgoaW50KHYpICUgKFQgKiBDKSkgLy8gQywgKGludCh2KSAlIChUICogQykpICUgQykgICAjIC0+ICh0aW1lLCBzZW5zb3IpCnJlbCwgZml4ID0gdG9faW1hZ2VfcmVsZWFzZWQoeCksIHRvX2ltYWdlX2ZpeGVkKHgpCmNoZWNrKCdDMSBib3RoIGxheW91dHMgZ2l2ZSAoQiw5LDEsMTI4KScsIHJlbC5zaGFwZSA9PSBmaXguc2hhcGUgPT0gKEIsIEMsIDEsIFQpLCBzdHIodHVwbGUocmVsLnNoYXBlKSkpCm9rX2ZpeCA9IGFsbChkZWMoZml4WzAsIGMsIDAsIHRdKSA9PSAodCwgYykgZm9yIGMgaW4gcmFuZ2UoQykgZm9yIHQgaW4gcmFuZ2UoVCkpCmNoZWNrKCdDMiBmaXhlZCBsYXlvdXQ6IGNoYW5uZWwgYyA9IHNlbnNvciBjLCB3aWR0aCB0ID0gdGltZSB0IChhbGwgMTE1MiBwb3NpdGlvbnMpJywgb2tfZml4KQpyb3cwID0gW2RlYyhyZWxbMCwgMCwgMCwgdF0pIGZvciB0IGluIHJhbmdlKDYpXQpzZW5zX2luX2tlcm5lbCA9IFtsZW4oe2RlYyhyZWxbMCwgYywgMCwgdCArIGtdKVsxXSBmb3IgayBpbiByYW5nZSg1KX0pIGZvciBjIGluIHJhbmdlKEMpIGZvciB0IGluIHJhbmdlKFQgLSA0KV0KY2hlY2soJ0MzIHJlbGVhc2VkIGxheW91dCBtaXhlcyBzZW5zb3JzOiBldmVyeSB3aWR0aC01IGNvbnYgd2luZG93IHNwYW5zIDUgZGlmZmVyZW50IHNlbnNvcnMnLAogICAgICBtaW4oc2Vuc19pbl9rZXJuZWwpID09IDUsIGYnZmlyc3QgcG9zaXRpb25zIG9mIGNoYW5uZWwgMCBhcyAodGltZSxzZW5zb3IpOiB7cm93MH0nKQpyZWxfY2hfc2Vuc29ycyA9IFtsZW4oe2RlYyhyZWxbMCwgYywgMCwgdF0pWzFdIGZvciB0IGluIHJhbmdlKFQpfSkgZm9yIGMgaW4gcmFuZ2UoQyldCmNoZWNrKCdDNCByZWxlYXNlZCAiY2hhbm5lbCIgaG9sZHMgYWxsIDkgc2Vuc29ycyBhbmQgfjE0IHRpbWUgc3RlcHMnLCByZWxfY2hfc2Vuc29ycyA9PSBbOV0gKiBDLAogICAgICBmJ3RpbWUgc3BhbiBvZiBjaGFubmVsIDA6IHtkZWMocmVsWzAsMCwwLDBdKVswXX0uLntkZWMocmVsWzAsMCwwLFQtMV0pWzBdfScpCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gQzUtQzc6IHBhdGNoZWQgbW9kZWwKbWFyZ3MgPSBhcmdwYXJzZS5OYW1lc3BhY2Uobm93X21vZGVsX25hbWU9J0dJTEUnLCBiYXRjaF9zaXplPUIsIG5fZXBvY2g9MSwgZGF0YXNldD0ndWNpaGFyJywgbl9mZWF0dXJlPTksIGxlbl9zdz0xMjgsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIG5fY2xhc3M9NiwgZF9BRT01MCwgbl9kb21haW5zPTUsIHhfZGltPTExNTIsIGF1eF9sb3NzX211bHRpcGxpZXJfeT0xMDAwLiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgYXV4X2xvc3NfbXVsdGlwbGllcl9kPTEwMDAuLCBiZXRhX2Q9MS4sIGJldGFfeD0wLiwgYmV0YV95PTEuLCB3ZWlnaHRfdHJ1ZT0xMDAwLiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgd2VpZ2h0X2ZhbHNlPTEwMDAuLCBkZXZpY2U9REVWLCB0YXJnZXRfZG9tYWluPScwJykKdG9yY2gubWFudWFsX3NlZWQoMCk7IG1fcmVsID0gR0lMRShtYXJncykudG8oREVWKQp0b3JjaC5tYW51YWxfc2VlZCgwKTsgbV9maXggPSBHSUxFKG1hcmdzKS50byhERVYpCnNhbWVfaW5pdCA9IGFsbCh0b3JjaC5lcXVhbChhLCBiKSBmb3IgYSwgYiBpbiB6aXAobV9yZWwuc3RhdGVfZGljdCgpLnZhbHVlcygpLCBtX2ZpeC5zdGF0ZV9kaWN0KCkudmFsdWVzKCkpKQpwYXRjaGVkID0gYXBwbHlfbGF5b3V0X2ZpeChtX2ZpeCkKY2hlY2soJ0M1IHBhdGNoIHRvdWNoZXMgb25seSB0aGUgZW5jb2RlcnMsIGluaXQgaWRlbnRpY2FsJywgc2FtZV9pbml0IGFuZCBwYXRjaGVkID09IFsncXpkJywgJ3F6eSddLCBzdHIocGF0Y2hlZCkpCm1fcmVsLmV2YWwoKTsgbV9maXguZXZhbCgpCnhyID0gdG9yY2gucmFuZG4oQiwgVCwgQywgMSwgZGV2aWNlPURFVikKeF9lcXVpdiA9IHhyLnBlcm11dGUoMCwgMiwgMywgMSkuY29udGlndW91cygpLnZpZXcoQiwgVCwgQywgMSkgICAgICAgIyBtZW1vcnkgc3VjaCB0aGF0IC52aWV3KCkgPT0gY29ycmVjdCBwZXJtdXRlCndpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgYSA9IG1fZml4LnF6eSh4cilbMF07IGIgPSBtX3JlbC5xenkoeF9lcXVpdilbMF07IGEyID0gbV9maXgucXpkKHhyKVswXTsgYjIgPSBtX3JlbC5xemQoeF9lcXVpdilbMF0KICAgIGMgPSBtX3JlbC5xenkoeHIpWzBdCmNoZWNrKCdDNiBwYXRjaGVkIGZvcndhcmQgPT0gcmVsZWFzZWQgZm9yd2FyZCBhcHBsaWVkIHRvIGNvcnJlY3RseSBsYWlkLW91dCBtZW1vcnkgKHF6eSBhbmQgcXpkKScsCiAgICAgIHRvcmNoLmFsbGNsb3NlKGEsIGIsIGF0b2w9MCwgcnRvbD0wKSBhbmQgdG9yY2guYWxsY2xvc2UoYTIsIGIyLCBhdG9sPTAsIHJ0b2w9MCkpCmNoZWNrKCdDNyBwYXRjaGVkIGZvcndhcmQgZGlmZmVycyBmcm9tIHJlbGVhc2VkIGZvcndhcmQgb24gdGhlIHNhbWUgaW5wdXQnLCBub3QgdG9yY2guYWxsY2xvc2UoYSwgYykpCm1fZml4LnRyYWluKCk7IG1fZml4Lnplcm9fZ3JhZCgpCmQgPSB0b3JjaC5yYW5kaW50KDAsIDUsIChCLCksIGRldmljZT1ERVYpOyB5ID0gdG9yY2gucmFuZGludCgwLCA2LCAoQiwpLCBkZXZpY2U9REVWKQpsLCBfID0gbV9maXgubG9zc19mdW5jdGlvbihkLCB4ciwgeSk7IGwuYmFja3dhcmQoKQpnMSA9IG1fZml4LnF6eS5jb252MVswXS53ZWlnaHQuZ3JhZDsgZzIgPSBtX2ZpeC5xemQuY29udjFbMF0ud2VpZ2h0LmdyYWQKY2hlY2soJ0M4IGxvc3MgZmluaXRlIGFuZCBncmFkaWVudHMgcmVhY2ggZmlyc3QgY29udiBvZiBxenkgYW5kIHF6ZCAoZml4ZWQgbGF5b3V0KScsCiAgICAgIHRvcmNoLmlzZmluaXRlKGwpIGFuZCBnMSBpcyBub3QgTm9uZSBhbmQgZzEuYWJzKCkuc3VtKCkgPiAwIGFuZCBnMi5hYnMoKS5zdW0oKSA+IDAsIGYnbG9zcyB7ZmxvYXQobCk6LjFmfScpCnhfcmVjID0gbV9maXgoZCwgeHIsIHkpWzBdCmNoZWNrKCdDOSBkZWNvZGVyIG91dHB1dCBzaGFwZSA9PSBpbnB1dCBzaGFwZSAocmVjb25zdHJ1Y3Rpb24gdGFyZ2V0IHVuY2hhbmdlZCknLCB4X3JlYy5zaGFwZSA9PSB4ci5zaGFwZSkKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBBMS1BNjogYXVnbWVudGF0aW9uCmcgPSB0b3JjaC5HZW5lcmF0b3IoKTsgZy5tYW51YWxfc2VlZCgxKQp4YSA9IHRvcmNoLnJhbmRuKDY0LCBULCBDLCAxLCBnZW5lcmF0b3I9ZykgKiAwLjIKeGFbLi4uLCA2OjksIDBdICs9IHRvcmNoLnRlbnNvcihbMS4wLCAwLjAsIDAuMF0pICAgICAgICAgICAgICAgICAgICAgIyB0b3RhbF9hY2MgPSBib2R5X2FjYyArIGdyYXZpdHkKeGFbLi4uLCA2OjksIDBdICs9IHhhWy4uLiwgMDozLCAwXQphdWcgPSBQZXJzb25TdHlsZUF1ZyhzZWVkPTMpCm4gPSB4YS5zaGFwZVswXQphcHBseSwgcywgYXhpcywgYW5nLCBuYSwgbmd5ID0gYXVnLmRyYXdzKG4sIFQpCiMgcm90YXRpb24gb25seSAocz0xLCBubyBub2lzZSwgYWxsIGFwcGxpZWQpOiBub3JtcyBvZiBldmVyeSB0cmlhZCBhbmQgb2YgZ3Jhdml0eSBwcmVzZXJ2ZWQKb3V0ID0gUGVyc29uU3R5bGVBdWcudHJhbnNmb3JtKHhhLCB0b3JjaC5vbmVzKG4sIGR0eXBlPXRvcmNoLmJvb2wpLCB0b3JjaC5vbmVzKG4sIGR0eXBlPXRvcmNoLmZsb2F0NjQpLCBheGlzLCBhbmcsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3JjaC56ZXJvc19saWtlKG5hKSwgdG9yY2guemVyb3NfbGlrZShuZ3kpKQpucm0gPSBsYW1iZGEgdiwgc2w6IHZbLi4uLCBzbCwgMF0ubm9ybShkaW09LTEpCm9rbiA9IGFsbCh0b3JjaC5hbGxjbG9zZShucm0ob3V0LCBzbCksIG5ybSh4YSwgc2wpLCBhdG9sPTFlLTUpIGZvciBzbCBpbiAoc2xpY2UoMCwgMyksIHNsaWNlKDMsIDYpLCBzbGljZSg2LCA5KSkpCmdyYXZfaW4gPSAoeGFbLi4uLCA2OjksIDBdIC0geGFbLi4uLCAwOjMsIDBdKS5ub3JtKGRpbT0tMSk7IGdyYXZfb3V0ID0gKG91dFsuLi4sIDY6OSwgMF0gLSBvdXRbLi4uLCAwOjMsIDBdKS5ub3JtKGRpbT0tMSkKY2hlY2soJ0ExIHJvdGF0aW9uIHByZXNlcnZlcyBldmVyeSB0cmlhZCBub3JtIGFuZCB0aGUgZ3Jhdml0eSBtYWduaXR1ZGUnLCBva24gYW5kIHRvcmNoLmFsbGNsb3NlKGdyYXZfaW4sIGdyYXZfb3V0LCBhdG9sPTFlLTUpKQpSID0gX3JvdGF0aW9ucyhheGlzLCBhbmcpCmNoZWNrKCdBMiByb3RhdGlvbnMgYXJlIHByb3BlciAoUiBSXlQgPSBJLCBkZXQgPSArMSkgYW5kIHdpdGhpbiArLTE1IGRlZycsCiAgICAgIHRvcmNoLmFsbGNsb3NlKFIgQCBSLnRyYW5zcG9zZSgxLCAyKSwgdG9yY2guZXllKDMsIGR0eXBlPVIuZHR5cGUpLmV4cGFuZF9hcyhSKSwgYXRvbD0xZS0xMCkKICAgICAgYW5kIHRvcmNoLmFsbGNsb3NlKHRvcmNoLmRldChSKSwgdG9yY2gub25lcyhuLCBkdHlwZT1SLmR0eXBlKSwgYXRvbD0xZS0xMCkKICAgICAgYW5kIGZsb2F0KGFuZy5hYnMoKS5tYXgoKSkgPD0gbWF0aC5yYWRpYW5zKDE1KSArIDFlLTEyLCBmJ21heCBhbmdsZSB7bWF0aC5kZWdyZWVzKGZsb2F0KGFuZy5hYnMoKS5tYXgoKSkpOi4yZn0gZGVnJykKb3V0MiA9IFBlcnNvblN0eWxlQXVnLnRyYW5zZm9ybSh4YSwgYXBwbHksIHMsIGF4aXMsIGFuZywgbmEsIG5neSkKc2FtZV9vZmYgPSB0b3JjaC5lcXVhbChvdXQyW35hcHBseV0sIHhhW35hcHBseV0pOyBjaGFuZ2VkX29uID0gbm90IHRvcmNoLmFsbGNsb3NlKG91dDJbYXBwbHldLCB4YVthcHBseV0pCmNoZWNrKCdBMyB3aW5kb3dzIG5vdCBzZWxlY3RlZCBhcmUgYml0LWlkZW50aWNhbDsgc2VsZWN0ZWQgd2luZG93cyBjaGFuZ2UnLCBzYW1lX29mZiBhbmQgY2hhbmdlZF9vbiwKICAgICAgZid7aW50KGFwcGx5LnN1bSgpKX0ve259IHNlbGVjdGVkJykKIyBzY2FsaW5nIG9ubHk6IGdyYXZpdHkgY29tcG9uZW50IHVuY2hhbmdlZCwgYm9keSBwYXJ0cyBzY2FsZWQgYnkgcwpvdXQzID0gUGVyc29uU3R5bGVBdWcudHJhbnNmb3JtKHhhLCB0b3JjaC5vbmVzKG4sIGR0eXBlPXRvcmNoLmJvb2wpLCBzLCBheGlzLCB0b3JjaC56ZXJvc19saWtlKGFuZyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdG9yY2guemVyb3NfbGlrZShuYSksIHRvcmNoLnplcm9zX2xpa2Uobmd5KSkKY2hlY2soJ0E0IHNjYWxpbmcgY2hhbmdlcyBib2R5X2FjYy9ib2R5X2d5cm8gYnkgcyBhbmQgbGVhdmVzIGdyYXZpdHkgKHRvdGFsLWJvZHkpIHVuY2hhbmdlZCcsCiAgICAgIHRvcmNoLmFsbGNsb3NlKG91dDNbLi4uLCAwOjMsIDBdLCAoc1s6LCBOb25lLCBOb25lXSAqIHhhWy4uLiwgMDozLCAwXS5kb3VibGUoKSkuZmxvYXQoKSwgYXRvbD0xZS01KQogICAgICBhbmQgdG9yY2guYWxsY2xvc2Uob3V0M1suLi4sIDY6OSwgMF0gLSBvdXQzWy4uLiwgMDozLCAwXSwgeGFbLi4uLCA2OjksIDBdIC0geGFbLi4uLCAwOjMsIDBdLCBhdG9sPTFlLTUpCiAgICAgIGFuZCBmbG9hdChzLm1pbigpKSA+PSAwLjcgYW5kIGZsb2F0KHMubWF4KCkpIDw9IDEuMykKYTEsIGEyID0gUGVyc29uU3R5bGVBdWcoNSksIFBlcnNvblN0eWxlQXVnKDUpCnIxID0gYTEoeGEpOyByMiA9IGEyKHhhLmNsb25lKCkgKiAwICsgNykgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBkaWZmZXJlbnQgY29udGVudCwgc2FtZSBzZWVkCnN0X2VxdWFsID0gdG9yY2guZXF1YWwoYTEuZy5nZXRfc3RhdGUoKSwgYTIuZy5nZXRfc3RhdGUoKSkKcjMgPSBQZXJzb25TdHlsZUF1Zyg1KSh4YSkKY2hlY2soJ0E1IGRldGVybWluaXN0aWMgcGVyIHNlZWQ7IHJhbmRvbSBkcmF3cyBkbyBub3QgZGVwZW5kIG9uIHRoZSBkYXRhJywgdG9yY2guZXF1YWwocjEsIHIzKSBhbmQgc3RfZXF1YWwpCmNoZWNrKCdBNiBnbG9iYWwgdG9yY2ggUk5HIHVudG91Y2hlZCBieSBhdWdtZW50YXRpb24nLAogICAgICAobGFtYmRhIHMwOiAoUGVyc29uU3R5bGVBdWcoOSkoeGEpLCB0b3JjaC5lcXVhbChzMCwgdG9yY2guZ2V0X3JuZ19zdGF0ZSgpKSlbMV0pKHRvcmNoLmdldF9ybmdfc3RhdGUoKSkpCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gUjEtUjQ6IHJ1bm5lciAoZmFrZSBkYXRhLCBzaG9ydCkKUFkgPSBzeXMuZXhlY3V0YWJsZQpkZWYgcnVuKHNjcmlwdCwgKmEpOgogICAgciA9IHN1YnByb2Nlc3MucnVuKFtQWSwgc2NyaXB0LCAqYV0sIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSkKICAgIGlmIHIucmV0dXJuY29kZSBub3QgaW4gKDAsIDMpOiBwcmludChyLnN0ZG91dFstMTUwMDpdLCByLnN0ZGVyclstMTUwMDpdKQogICAgcmV0dXJuIHIucmV0dXJuY29kZQpSVU4zID0gb3MucGF0aC5qb2luKFMzLCAncnVuX3N0YWdlMy5weScpOyBSVU4yID0gb3MucGF0aC5qb2luKENPREUsICdzdGFnZTJfcmVmJywgJ3J1bl9zdGFnZTIucHknKQpURVNUID0gJ3J1bnNfdGVzdHMnCmZvciByb290IGluIChvcy5wYXRoLmpvaW4oUzMsIFRFU1QpLCBvcy5wYXRoLmpvaW4oQ09ERSwgJ3N0YWdlMl9yZWYnLCBURVNUKSk6CiAgICBpZiBvcy5wYXRoLmlzZGlyKHJvb3QpOiBzaHV0aWwucm10cmVlKHJvb3QpCmNvbW1vbiA9IFsnLS10YXJnZXQnLCAnMScsICctLXNlZWQnLCAnMicsICctLWVwb2NocycsICczJywgJy0tZmFrZScsICctLW91dF9yb290JywgVEVTVCwgJy0tbm9fbGFzdF9ja3B0J10KcmMgPSBbcnVuKFJVTjMsICctLWFybScsIGEsICpjb21tb24pIGZvciBhIGluICgnaWVfb2ZmJywgJ2ZpeCcsICdhdWcnLCAnZml4X2F1ZycpXQpyYy5hcHBlbmQocnVuKFJVTjIsICctLXZhcmlhbnQnLCAnaWVfb2ZmJywgKmNvbW1vbikpCmNoZWNrKCdSMCBhbGwgc2hvcnQgcnVucyBjb21wbGV0ZWQnLCByYyA9PSBbMF0gKiA1LCBzdHIocmMpKQpsZCA9IGxhbWJkYSByb290LCBhcm0sIGY6IGpzb24ubG9hZChvcGVuKG9zLnBhdGguam9pbihyb290LCBURVNULCAndWNpaGFyJywgYXJtLCAndGFyZ2V0MScsICdzZWVkMicsIGYpKSkKbTMgPSB7YTogbGQoUzMsIGEsICdtZXRyaWNzLmpzb24nKSBmb3IgYSBpbiAoJ2llX29mZicsICdmaXgnLCAnYXVnJywgJ2ZpeF9hdWcnKX0KYzMgPSB7YTogbGQoUzMsIGEsICdjb25maWcuanNvbicpIGZvciBhIGluICgnaWVfb2ZmJywgJ2ZpeCcsICdhdWcnLCAnZml4X2F1ZycpfQptMiA9IGxkKG9zLnBhdGguam9pbihDT0RFLCAnc3RhZ2UyX3JlZicpLCAnaWVfb2ZmJywgJ21ldHJpY3MuanNvbicpCmltcG9ydCBjc3YKaGlzdCA9IGxhbWJkYSByb290LCBhcm06IGxpc3QoY3N2LkRpY3RSZWFkZXIob3Blbihvcy5wYXRoLmpvaW4ocm9vdCwgVEVTVCwgJ3VjaWhhcicsIGFybSwgJ3RhcmdldDEnLCAnc2VlZDInLCAnaGlzdG9yeS5jc3YnKSkpKQpoMywgaDIgPSBoaXN0KFMzLCAnaWVfb2ZmJyksIGhpc3Qob3MucGF0aC5qb2luKENPREUsICdzdGFnZTJfcmVmJyksICdpZV9vZmYnKQprZXlzID0gKCd0cmFpbl9sb3NzX3Blcl9zYW1wbGUnLCAndmFsX2FjY19tZWFuJywgJ3ZhbF9jZV95X21lYW4nLCAndGFyZ2V0X2FjY19ESUFHJykKc2FtZV9oaXN0ID0gYWxsKHIzW2tdID09IHIyW2tdIGZvciByMywgcjIgaW4gemlwKGgzLCBoMikgZm9yIGsgaW4ga2V5cykgYW5kIGxlbihoMykgPT0gbGVuKGgyKQpwbnVtID0gbGFtYmRhIG06IHtrOiB2IGZvciBrLCB2IGluIG1bJ3Byb2Jlc19zZWxlY3RlZF9lcG9jaCddLml0ZW1zKCkgaWYgayAhPSAnbm90ZSd9ICAgIyAnbm90ZScgdGV4dCB3YXMgcmV3b3JkZWQKc2FtZV9yZXMgPSBhbGwobTNbJ2llX29mZiddW2tdID09IG0yW2tdIGZvciBrIGluICgndGFyZ2V0X2F0X3NlbGVjdGVkX2Vwb2NoJywgJ3RhcmdldF9hdF9sYXN0X2Vwb2NoJykpIFwKICAgIGFuZCBwbnVtKG0zWydpZV9vZmYnXSkgPT0gcG51bShtMikKY2hlY2soJ1IxIHN0YWdlLTMgY29udHJvbCBhcm0gPT0gc3RhZ2UtMiBydW5uZXIgaWVfb2ZmIChoaXN0b3J5LCB0YXJnZXQgbWV0cmljcywgcHJvYmVzKScsIHNhbWVfaGlzdCBhbmQgc2FtZV9yZXMpCmNoZWNrKCdSMiBpZGVudGljYWwgaW5pdGlhbCBwYXJhbWV0ZXJzIGluIGFsbCBmb3VyIGFybXMnLAogICAgICBsZW4oe2MzW2FdWydpbml0aWFsX3BhcmFtZXRlcl9oYXNoJ10gZm9yIGEgaW4gYzN9KSA9PSAxLCBjM1snaWVfb2ZmJ11bJ2luaXRpYWxfcGFyYW1ldGVyX2hhc2gnXSkKZGlmZiA9IGxhbWJkYSBhOiBoaXN0KFMzLCBhKVstMV1bJ3RyYWluX2xvc3NfcGVyX3NhbXBsZSddICE9IGgzWy0xXVsndHJhaW5fbG9zc19wZXJfc2FtcGxlJ10KY2hlY2soJ1IzIGZpeCAvIGF1ZyAvIGZpeF9hdWcgZWFjaCBjaGFuZ2UgdHJhaW5pbmcgcmVsYXRpdmUgdG8gdGhlIGNvbnRyb2wnLCBhbGwoZGlmZihhKSBmb3IgYSBpbiAoJ2ZpeCcsICdhdWcnLCAnZml4X2F1ZycpKSkKIyByZXN1bWU6IGZpeF9hdWcgaW50ZXJydXB0ZWQgYWZ0ZXIgZXBvY2ggMSB0aGVuIHJlc3VtZWQgPT0gdW5pbnRlcnJ1cHRlZApURVNUMiA9ICdydW5zX3Rlc3RzX3Jlc3VtZScKaWYgb3MucGF0aC5pc2Rpcihvcy5wYXRoLmpvaW4oUzMsIFRFU1QyKSk6IHNodXRpbC5ybXRyZWUob3MucGF0aC5qb2luKFMzLCBURVNUMikpCmMyID0gWyctLXRhcmdldCcsICcxJywgJy0tc2VlZCcsICcyJywgJy0tZXBvY2hzJywgJzMnLCAnLS1mYWtlJywgJy0tb3V0X3Jvb3QnLCBURVNUMiwgJy0tbm9fbGFzdF9ja3B0J10Kcl9hID0gcnVuKFJVTjMsICctLWFybScsICdmaXhfYXVnJywgKmMyLCAnLS1zdG9wX2FmdGVyX2Vwb2NoJywgJzEnKTsgcl9iID0gcnVuKFJVTjMsICctLWFybScsICdmaXhfYXVnJywgKmMyKQptcmVzID0ganNvbi5sb2FkKG9wZW4ob3MucGF0aC5qb2luKFMzLCBURVNUMiwgJ3VjaWhhcicsICdmaXhfYXVnJywgJ3RhcmdldDEnLCAnc2VlZDInLCAnbWV0cmljcy5qc29uJykpKQpocmVzID0gbGlzdChjc3YuRGljdFJlYWRlcihvcGVuKG9zLnBhdGguam9pbihTMywgVEVTVDIsICd1Y2loYXInLCAnZml4X2F1ZycsICd0YXJnZXQxJywgJ3NlZWQyJywgJ2hpc3RvcnkuY3N2JykpKSkKaHVuID0gaGlzdChTMywgJ2ZpeF9hdWcnKQpjaGVjaygnUjQgaW50ZXJydXB0ZWQrcmVzdW1lZCBmaXhfYXVnID09IHVuaW50ZXJydXB0ZWQgKGhpc3RvcnkgYW5kIHRhcmdldCBtZXRyaWNzKScsCiAgICAgIHJfYSA9PSAzIGFuZCByX2IgPT0gMCBhbmQgbXJlc1sncmVzdW1lJ11bJ3Jlc3VtZWRfYWZ0ZXJfZXBvY2gnXSA9PSAxCiAgICAgIGFuZCBhbGwoYVtrXSA9PSBiW2tdIGZvciBhLCBiIGluIHppcChocmVzLCBodW4pIGZvciBrIGluIGtleXMpCiAgICAgIGFuZCBtcmVzWyd0YXJnZXRfYXRfc2VsZWN0ZWRfZXBvY2gnXSA9PSBtM1snZml4X2F1ZyddWyd0YXJnZXRfYXRfc2VsZWN0ZWRfZXBvY2gnXSkKcHJpbnQoJ2RldmljZScsIERFViwgJ3wgdG9yY2gnLCB0b3JjaC5fX3ZlcnNpb25fXykKcHJpbnQoJ0FMTCBQQVNTRUQnIGlmIGFsbChyZXN1bHRzKSBlbHNlIGYnRkFJTEVEOiB7cmVzdWx0cy5jb3VudChGYWxzZSl9IG9mIHtsZW4ocmVzdWx0cyl9JykK"}}')
for name, item in PAYLOAD.items():
    data = base64.b64decode(item['b64']); assert hashlib.sha256(data).hexdigest() == item['sha256'], name
    path = os.path.join(PKG, name); os.makedirs(os.path.dirname(path), exist_ok=True); open(path, 'wb').write(data)
print(len(PAYLOAD), 'files written and verified')
CODE = os.path.join(PKG, 'code'); S3 = os.path.join(CODE, 'stage3'); OUT = 'runs_p3'; ROOT = os.path.join(S3, OUT); os.chdir(CODE)

In [ ]:
# 3. Download UCI HAR, verify SHA-256, arrange layout
import urllib.request, zipfile, shutil, io
URL, SHA = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip', 'c00b803081a5c797cd5e4b83700a9810b38d53d9d84e01917e090e1fdbc81031'
raw = urllib.request.urlopen(URL, timeout=300).read(); h = hashlib.sha256(raw).hexdigest(); print('sha256 ok:', h == SHA); assert h == SHA
inner = zipfile.ZipFile(io.BytesIO(zipfile.ZipFile(io.BytesIO(raw)).read('UCI HAR Dataset.zip'))); inner.extractall('/content/uci_raw')
src = '/content/uci_raw/UCI HAR Dataset'
for s in ('train', 'test'):
    dst = os.path.join(CODE, 'data', 'ucihar', s, 'InertialSignals'); os.makedirs(dst, exist_ok=True)
    for f in os.listdir(os.path.join(src, s, 'Inertial Signals')): shutil.copy(os.path.join(src, s, 'Inertial Signals', f), dst)
    for f in ('y_%s.txt' % s, 'subject_%s.txt' % s): shutil.copy(os.path.join(src, s, f), os.path.join(CODE, 'data', 'ucihar', s))
n = sum(len(fs) for _, _, fs in os.walk(os.path.join(CODE, 'data', 'ucihar'))); print(n, 'files (expect 22)'); assert n == 22

In [ ]:
# 4. Tests (must end with ALL PASSED)
r = subprocess.run([sys.executable, 'stage3/checks/test_stage3.py'], capture_output=True, text=True)
print(r.stdout[-4000:]); print(r.stderr[-1500:]); assert 'ALL PASSED' in r.stdout, 'TESTS FAILED - stop here'

In [ ]:
# 5. Helpers: 3 runs in parallel on the GPU (each run is deterministic on its own); skip completed runs
import time, glob
from concurrent.futures import ThreadPoolExecutor
from google.colab import files
ANCHOR = {'selected_epoch': 99, 'accuracy': 74.06340057636888, 'weighted_f1': 73.92573013405048}
ARMS = ('fix', 'aug', 'fix_aug'); TARGETS = ('0', '1', '2', '3', '4')
def run_one(a, t, seed):
    t0 = time.time()
    r = subprocess.run([sys.executable, 'stage3/run_stage3.py', '--arm', a, '--target', t, '--seed', str(seed),
                        '--out_root', OUT, '--no_last_ckpt'], capture_output=True, text=True)
    lines = [l for l in r.stdout.splitlines() if l.startswith(('DONE', 'SKIP', 'RESUMED', 'NOT RESUMED'))]
    print(f'seed {seed} target {t} {a:8s} exit {r.returncode} {time.time()-t0:5.0f}s | ' + ' || '.join(l.split(' -> ')[0] for l in lines), flush=True)
    if r.returncode != 0:
        d = os.path.join(ROOT, 'ucihar', a, f'target{t}', f'seed{seed}'); os.makedirs(d, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        open(os.path.join(d, f'FAILED_{stamp}_stderr.txt'), 'w').write(r.stderr); open(os.path.join(d, f'FAILED_{stamp}_stdout.txt'), 'w').write(r.stdout)
        print('   STDERR tail:', r.stderr.strip().splitlines()[-3:], flush=True)
    return (a, t, r.returncode)
def run_seed(seed, workers=3):
    t0 = time.time()
    with ThreadPoolExecutor(workers) as ex:
        res = list(ex.map(lambda at: run_one(at[0], at[1], seed), [(a, t) for t in TARGETS for a in ARMS]))
    fails = [x for x in res if x[2] != 0]; print('FAILED RUNS:', fails if fails else 'none', f'| seed wall time {time.time()-t0:.0f}s')
def package_seed(seed, extra=()):
    dirs = [os.path.join(ROOT, 'ucihar', a, f'target{t}', f'seed{seed}') for a in ARMS for t in TARGETS] + list(extra)
    done = [d for d in dirs if os.path.exists(os.path.join(d, 'metrics.json'))]
    z = f'/content/p3_t4_seed{seed}.zip'; man = f'MANIFEST_p3_seed{seed}.sha256'; lines = []
    with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as Z:
        for d in dirs:
            for f in sorted(glob.glob(os.path.join(d, '*'))):
                if f.endswith(('resume_state.pt', '.tmp')): continue
                arc = os.path.relpath(f, S3).replace(os.sep, '/')
                lines.append(f'{hashlib.sha256(open(f, "rb").read()).hexdigest()}  {arc}'); Z.write(f, arc)
        Z.writestr(man, '\n'.join(lines) + '\n')
    Z = zipfile.ZipFile(z); listed = dict(l.split('  ', 1)[::-1] for l in Z.read(man).decode().splitlines() if l.strip())
    bad = [a for a in Z.namelist() if a != man and hashlib.sha256(Z.read(a)).hexdigest() != listed.get(a)]
    print(z, len(listed), 'files', round(os.path.getsize(z)/2**20, 1), 'MB | verified', 'OK' if not bad else f'BAD {bad}', f'| complete runs {len(done)}/{len(dirs)}')
    return z

In [ ]:
# 6. Reproducibility anchor: control arm (ie_off) target 0 seed 0 must equal the stage-1 T4 value exactly
r = run_one('ie_off', '0', 0)
m = json.load(open(os.path.join(ROOT, 'ucihar', 'ie_off', 'target0', 'seed0', 'metrics.json')))
got = {'selected_epoch': m['selection']['selected_epoch'], 'accuracy': m['target_at_selected_epoch']['accuracy'], 'weighted_f1': m['target_at_selected_epoch']['weighted_f1']}
print('ANCHOR expected', ANCHOR); print('ANCHOR obtained', got); print('ANCHOR EXACT MATCH:', got == ANCHOR)

In [ ]:
# 7. SEED 0: 15 runs (3 arms x 5 targets, 3 in parallel), package + verify + download. WAIT for the download.
run_seed(0)
z = package_seed(0, extra=[os.path.join(ROOT, 'ucihar', 'ie_off', 'target0', 'seed0')])
files.download(z)

In [ ]:
# 8. SEED 1: 15 runs (3 arms x 5 targets, 3 in parallel), package + verify + download. WAIT for the download.
run_seed(1)
z = package_seed(1)
files.download(z)

In [ ]:
# 9. SEED 2: 15 runs (3 arms x 5 targets, 3 in parallel), package + verify + download. WAIT for the download.
run_seed(2)
z = package_seed(2)
files.download(z)

In [ ]:
# 10. SEED 3: 15 runs (3 arms x 5 targets, 3 in parallel), package + verify + download. WAIT for the download.
run_seed(3)
z = package_seed(3)
files.download(z)

In [ ]:
# 11. SEED 4: 15 runs (3 arms x 5 targets, 3 in parallel), package + verify + download. WAIT for the download.
run_seed(4)
z = package_seed(4)
files.download(z)